<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/kol_network_stance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# @title Install — jalankan sekali di awal
import os, shutil, sys, importlib.util
import torch
from google.colab import userdata

try:
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception as e:
    raise RuntimeError(
        f"Secret GH_TOKEN belum bisa dibaca ({type(e).__name__}). Cara memperbaiki:\n"
        "  1. Klik ikon 🔑 (Secrets) di sidebar kiri Colab\n"
        "  2. Klik 'Add new secret'. Name: GH_TOKEN, Value: token GitHub kamu\n"
        "     (token harus punya akses ke repo azmkto/GREAT-Tools; minta ke admin kalau belum punya)\n"
        "  3. Nyalakan toggle 'Notebook access' di baris GH_TOKEN\n"
        "  4. Jalankan ulang cell ini"
    ) from None

REPO_DIR = "/content/GREAT-Tools"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q https://{GH_TOKEN}@github.com/azmkto/GREAT-Tools.git {REPO_DIR}
if not os.path.isdir(f"{REPO_DIR}/great"):
    raise RuntimeError("Gagal download repo GREAT-Tools. Token GH_TOKEN salah, kedaluwarsa, "
                       "atau tidak punya akses ke repo azmkto/GREAT-Tools. Perbarui secret-nya lalu jalankan ulang.")

# torch, pandas, tqdm bawaan Colab dipakai apa adanya (torch-nya sudah versi GPU).
# Meng-install ulang paket yang sudah ter-import = wajib restart, jadi itu yang dihindari.
PACKAGES = ["emoji", "openpyxl", "ftfy", "gliner", "transformers", "networkx"]
already_loaded = [p for p in [*PACKAGES, "llama_cpp"] if p in sys.modules]
PIP_ARGS = " ".join(PACKAGES)
!pip install -q -U {PIP_ARGS}

# LLM lokal (Step 7): llama-cpp-python versi prebuilt sesuai CUDA, sama seperti topic_modelling.ipynb
HAS_GPU = torch.cuda.is_available()
cuda_major, cuda_minor = (torch.version.cuda or "0.0").split(".")[:2]
if HAS_GPU and cuda_major != "12":
    raise RuntimeError(f"CUDA {torch.version.cuda}: belum ada llama-cpp-python prebuilt. Ganti runtime ke T4 GPU.")
CU_TAG = f"cu12{min(int(cuda_minor), 4)}" if HAS_GPU else "cpu"   # abetlen menyediakan cu121..cu124 + cpu
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{CU_TAG}
if not HAS_GPU:
    print("⚠️ Tidak ada GPU: GLiNER dan LLM lokal akan SANGAT lambat. Runtime -> Change runtime type -> T4 GPU.")

missing = [m for m in ["torch", "tqdm", "llama_cpp", *PACKAGES] if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Paket gagal terpasang: {missing}. Scroll ke atas, cari error pip-nya.")

if already_loaded:
    # hanya terjadi kalau cell ini dijalankan dua kali di session yang sama
    print(f"[PERLU RESTART] {already_loaded} sudah ter-import sebelum di-update. "
          "Klik Runtime -> Restart session, lalu lanjut dari Cell ② (tidak perlu jalankan cell ini lagi).")
else:
    print(f"[OK] Semua paket siap ({CU_TAG}), tidak perlu restart. Lanjut: isi form di Cell ②, "
          "klik Cell ②, lalu Runtime -> Run cell and below.")

[PERLU RESTART] ['emoji', 'openpyxl', 'networkx'] sudah ter-import sebelum di-update. Klik Runtime -> Restart session, lalu lanjut dari Cell ② (tidak perlu jalankan cell ini lagi).


## Import Library

In [13]:
import json, os, random, re, sys, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import emoji
import networkx as nx
import pandas as pd
from google.colab import ai, drive
from IPython.display import display
from tqdm.auto import tqdm

# restart runtime menghapus sys.path & variabel, jadi repo didaftarkan ulang di sini
REPO_DIR = "/content/GREAT-Tools"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from great import EXPORT_COLUMNS, clean_for_bert, clean_for_ner, is_media_account

tqdm.pandas()

print("great loaded from:", sys.modules["great"].__file__)
print("Commit terakhir  :", os.popen(f"git -C {REPO_DIR} rev-parse --short HEAD").read().strip())
print("Model Colab AI   :", ai.list_models())

great loaded from: /content/GREAT-Tools/great/__init__.py
Commit terakhir  : ce6e44a
Model Colab AI   : ['google/gemini-2.5-flash', 'google/gemini-2.5-flash-lite', 'google/gemini-2.5-pro', 'google/gemini-3.1-pro-preview', 'google/gemini-3.5-flash']


# Colab AI

In [14]:
AI_MODEL = "google/gemini-2.5-flash"   # pilih dari ai.list_models()
SLEEP_BETWEEN_CALLS = 4
api_call_count = 0


def is_quota_error(e):
    msg = str(e).lower()
    return "429" in msg or "resource_exhausted" in msg or "quota" in msg


def parse_json_text(text):
    """JSON pertama di jawaban model (Colab AI tidak punya mode JSON)."""
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    if not starts:
        raise ValueError("tidak ada JSON di jawaban model")
    return json.JSONDecoder().raw_decode(text[min(starts):])[0]


def ask_json(prompt, parse=lambda x: x, retry=4):
    """Colab AI -> JSON -> parse(). Kuota habis langsung di-raise; None kalau semua retry gagal."""
    global api_call_count
    for attempt in range(retry):
        try:
            api_call_count += 1
            return parse(parse_json_text(ai.generate_text(prompt, model_name=AI_MODEL)))
        except Exception as e:
            if is_quota_error(e):
                raise
            wait = (20 if "503" in str(e) or "UNAVAILABLE" in str(e) else 5) * (attempt + 1)
            print(f"[WARN] AI gagal ({attempt + 1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return None


def order_by_index(parsed, n):
    """List objek ber-field 'index' (1..n) -> list urut. Error kalau ada index yang hilang."""
    by_idx = {int(it["index"]): it for it in parsed if isinstance(it, dict) and "index" in it}
    if set(by_idx) != set(range(1, n + 1)):
        raise ValueError(f"Index hasil tidak lengkap: dapat {sorted(by_idx)}, harus 1..{n}")
    return [by_idx[i] for i in range(1, n + 1)]

In [15]:
PROJECT = "prabowo_vladivostok"
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"
STANCE_TARGET = "Presiden Prabowo dan pemerintah"
SOURCES_TEXT = "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"
REGENERATE_TOPIC_CONFIG = False

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
TOPIC = TOPIC.strip()
STANCE_TARGET = STANCE_TARGET.strip()
SOURCES = [s for s in re.split(r"[,\s]+", SOURCES_TEXT) if s]

problems = [msg for ok, msg in [(PROJECT, "PROJECT masih kosong"),
                                (TOPIC, "TOPIC masih kosong"),
                                (STANCE_TARGET, "STANCE_TARGET masih kosong"),
                                (SOURCES, "SOURCES_TEXT masih kosong (isi link Google Sheets)")] if not ok]
if problems:
    raise ValueError("Form belum lengkap: " + "; ".join(problems))
print(f"[OK] PROJECT = {PROJECT}\n     TOPIC   = {TOPIC}\n     STANCE  = sikap terhadap {STANCE_TARGET}\n"
      f"     {len(SOURCES)} file sumber")

[OK] PROJECT = prabowo_vladivostok
     TOPIC   = Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin
     STANCE  = sikap terhadap Presiden Prabowo dan pemerintah
     1 file sumber


# Konfigurasi
Edit bagian ini kalau nama kolom, daftar tema, atau bobot ranking berubah.
Semua cell di bawah memakai variabel dari sini.


In [16]:
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive"

COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_MEDIA = "Media"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]   # Location tidak dipakai di sini

GLINER_MODEL_NAME = "urchade/gliner_medium-v2.1"
GLINER_LABELS = ["lokasi", "instansi", "tokoh"]   # nama label di seluruh notebook
# GLiNER dilatih dengan label bahasa Inggris -> model diberi label Inggris, hasilnya dipetakan ke nama di atas
GLINER_PROMPT_LABELS = {"location": "lokasi", "organization": "instansi", "person": "tokoh"}
GLINER_THRESHOLD = 0.5
# Label map IndoBERT tidak di-hardcode, diambil dari config model di Step 6A.
INDOBERT_MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"

# LLM lokal untuk summary + stance (Step 7), sama dengan GREAT-Tools/Notebook/topic_modelling.ipynb
LLM_MODEL_REPO = "faizonly5953/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF"
LLM_MODEL_FILE = "gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf"
LLM_N_CTX = 8192

# Skor volume (Step 5), jumlahnya 1.0
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15
# Skor jaringan (Step 5B), jumlahnya 1.0
NET_WEIGHT_PAGERANK = 0.5   # pengaruh yang mengalir: di-retweet/mention oleh akun yang juga berpengaruh
NET_WEIGHT_EIGEN = 0.5      # Eigenvector: terhubung dengan akun-akun yang sama-sama terhubung kuat
# Skor KOL final (Step 5B) = volume + jaringan, jumlahnya 1.0
KOL_WEIGHT_VOLUME = 0.5    # jumlah post asli + engagement + followers
KOL_WEIGHT_NETWORK = 0.5   # PageRank + Eigenvector

BATCH_SIZE_CLASSIFY = 40
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15

RANKING_MIN_CONFIDENCE = 0.6      # tema di bawah ini tidak ikut ranking & jaringan
SENTIMENT_REVIEW_THRESHOLD = 0.7  # IndoBERT di bawah ini dikirim ke AI
REVIEW_SAMPLE_RATE = 0.05         # sampel acak confidence tinggi yang tetap dicek AI
DEDUP_PER_AUTHOR = False          # True = post kembar (copas) dari author berbeda tetap dihitung

# Merge akun: satu akun/outlet bisa muncul dengan beberapa ejaan -> satu key (dari author_network_extractor)
ALIASES = {}   # key -> key, untuk merge yang terlewat aturan. Contoh: {"republikaonline": "republikacoid"}
TWO_PART_SLD = {"co", "or", "ac", "go", "web", "my", "sch", "net", "com"}


def canonical(name):
    """Key merge akun: '@KOMPASTVCOM', 'www.kompastv.com' -> 'kompastvcom'; 'prabowo.subianto' = '@Prabowo_Subianto'.
    Author kosong -> None (bukan 'nan'), supaya post tanpa Author tidak digabung jadi satu akun palsu."""
    if pd.isna(name) or not str(name).strip():
        return None
    s = re.sub(r"^(https?://)?(www\.)?@?", "", str(name).strip().lower()).split("/")[0]
    parts = s.split(".")
    if len(parts) > 2 and re.fullmatch(r"[a-z]{2,3}", parts[-1]):   # domain, bukan handle IG bertitik
        parts = parts[-3:] if parts[-2] in TWO_PART_SLD and len(parts[-1]) == 2 else parts[-2:]
    key = re.sub(r"[^a-z0-9]", "", "".join(parts))
    return ALIASES.get(key, key)


assert canonical("@KOMPASTVCOM") == canonical("www.kompastv.com") == "kompastvcom"
assert canonical("nasional.kompas.com") == canonical("@kompascom") == "kompascom"
assert canonical("https://www.republika.co.id/berita/x") == "republikacoid"
assert canonical("prabowo.subianto") == canonical("@Prabowo_Subianto") == "prabowosubianto"
assert canonical(float("nan")) is None and canonical("  ") is None

# Pending training data (dipakai bersama semua topik, tiap baris bawa field project)
PENDING_PATHS = {
    "ner": f"{DRIVE_DIR}/pending_ner_data.jsonl",
    "sentimen": f"{DRIVE_DIR}/pending_sentiment_data.jsonl",
    "tema": f"{DRIVE_DIR}/pending_tema_data.jsonl",
}
TOPIC_CONFIG_PATH = f"{DRIVE_DIR}/topic_config_{PROJECT}.json"
CHECKPOINT_PATH = f"{DRIVE_DIR}/checkpoint_4d_{PROJECT}.pkl"
SAVE_PATH_AFTER_THEME = f"{DRIVE_DIR}/df_clean_after_theme_{PROJECT}.pkl"
# Hasil (KOL, nodes, edges, sentimen, stance, ringkasan) TIDAK disimpan ke file, cukup ditampilkan penuh di notebook.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


def save_pending(task, rows):
    """Append ke pending. Hanya label dari AI/manual; output mentah model ditolak."""
    path = PENDING_PATHS[task]
    rows = [r for r in rows if r.get("text") and r.get("label_source") in ("gemini", "manual")]
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({**r, "task": task, "project": PROJECT, "run_date": str(date.today())},
                               ensure_ascii=False) + "\n")
    print(f"[pending] {task}: {len(rows)} baris ditulis ke {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Konfigurasi topik dari Colab AI

In [17]:
TOPIC_CONFIG_PROMPT = f"""Kamu analis media sosial Indonesia yang menyiapkan konfigurasi klasifikasi otomatis.
Topik yang dipantau: "{TOPIC}".
Tanggal hari ini: {date.today()}. Jangan menulis tahun di keyword kecuali yakin tahunnya benar.

Konfigurasi ini dipakai oleh pipeline berikut, jadi ikuti aturan tiap field dengan ketat:
1. Cuitan yang TIDAK mengandung satu pun `keywords` langsung dibuang sebagai "lainnya" tanpa dibaca AI.
   Keyword yang terlewat = cuitan relevan hilang diam-diam. Keyword terlalu umum = ribuan cuitan
   tidak relevan ikut diproses (lama & boros kuota).
2. Cuitan yang lolos dibaca AI dan diberi satu tema berdasarkan deskripsi di `themes`.
3. Nama entitas (tokoh, lokasi, instansi) yang ditemukan disatukan lewat `aliases`.
4. Akun di `media_accounts` DIKELUARKAN dari ranking top author.

PENTING: data mentah biasanya hasil monitoring yang LEBIH LUAS dari topik ini, misalnya SEMUA cuitan
yang menyebut tokoh utamanya (termasuk isu lain yang tidak berhubungan). Karena itu nama tokoh utama
saja BUKAN penanda topik ini.

=== ATURAN PER FIELD ===

themes (objek: nama_tema -> deskripsi)
- 2-4 tema yang membagi percakapan tentang topik ini secara jelas. Tiap cuitan relevan harus
  masuk tepat satu tema. Pilih pembagian berdasarkan sudut pandang percakapan (misal: agenda/acara,
  isu kebijakan/kerja sama, reaksi politik/geopolitik), bukan berdasarkan tokoh.
- Nama: snake_case, singkat, bahasa Indonesia (contoh: kerja_sama_ekonomi).
- Deskripsi satu string dengan format:
  "Masuk: <apa yang dibahas, 1 kalimat>. Contoh frasa: <8-15 frasa khas cuitan, termasuk bahasa
  gaul, singkatan, julukan>. Bukan: <yang sering tertukar dengan tema lain dan masuk ke tema mana>."
- Jangan buat tema "lainnya", "umum", atau "campuran". Itu sudah disediakan sistem.

keywords (list string)
- 20-60 kata/frasa, huruf kecil semua, dicocokkan sebagai frasa PERSIS dan KATA UTUH.
  Contoh: "putin" TIDAK cocok dengan "putinisme"; "prabowo putin" TIDAK cocok dengan "prabowo bertemu putin".
  Jadi utamakan kata tunggal yang khas, bukan frasa panjang.
- Isi dengan penanda yang KHAS topik ini: nama pihak lain, tempat, acara, dan instansi yang hampir
  hanya muncul saat membicarakan topik ini (ejaan Indonesia dan Inggris, singkatan, salah ketik umum,
  bentuk hashtag digabung tanpa #).
- DILARANG: nama tokoh utama sendirian (misal "prabowo", "presiden prabowo", julukannya), dan kata umum
  yang juga dipakai di isu lain (misal "investasi", "pertahanan", "kesepakatan", "perdamaian",
  "kunjungan kerja", "kebijakan luar negeri", "ketahanan pangan").

aliases (objek: variasi huruf kecil -> nama baku)
- Key: tulisan persis yang mungkin muncul di cuitan, huruf kecil (nama panggilan, singkatan,
  salah ketik, versi tanpa gelar, username tanpa @).
- Value: nama baku lengkap dengan kapitalisasi benar (contoh: "wowo" -> "Prabowo Subianto").
- Satu nama baku boleh punya banyak key. Jangan buat alias yang ambigu (satu key cocok ke dua hal,
  misal "ri").
- Minimal 15 entri, mencakup tokoh, lokasi, dan instansi.

media_accounts (list string)
- Username akun berita/media dan kanal resmi institusi yang khusus terkait topik ini
  (contoh: kantor berita negara asing, akun humas lembaga). Huruf kecil, tanpa @.
- JANGAN masukkan akun pribadi tokoh politik, buzzer, atau influencer: mereka justru
  harus ikut dianalisis sebagai author.
- Kalau tidak yakin username-nya ada, jangan dimasukkan. List kosong boleh.

=== CEK SEBELUM MENJAWAB ===
- Tidak ada keyword berupa nama tokoh utama sendirian atau kata umum.
- Setiap tema punya bagian "Bukan:" yang menyebut batas dengan tema lain.
- Semua key aliases dan semua keywords huruf kecil.

=== FORMAT JAWABAN ===
Jawab HANYA satu objek JSON, tanpa markdown, tanpa ```, tanpa teks sebelum atau sesudahnya:
{{"themes": {{"nama_tema": "Masuk: ... Contoh frasa: ... Bukan: ..."}},
  "keywords": ["...", "..."],
  "aliases": {{"variasi": "Nama Baku"}},
  "media_accounts": ["..."]}}"""

# config lama hanya dipakai kalau disusun untuk TOPIC yang sama persis
topic_config = None
if os.path.exists(TOPIC_CONFIG_PATH) and not REGENERATE_TOPIC_CONFIG:
    with open(TOPIC_CONFIG_PATH, encoding="utf-8") as f:
        cached = json.load(f)
    if cached.get("_topic") == TOPIC:
        topic_config = cached
        print(f"[INFO] Konfigurasi topik dibaca dari {TOPIC_CONFIG_PATH}")
    else:
        print("[INFO] TOPIC berbeda dari konfigurasi yang tersimpan -> AI menyusun ulang")

if topic_config is None:
    topic_config = ask_json(TOPIC_CONFIG_PROMPT)
    if not topic_config or not topic_config.get("themes"):
        raise RuntimeError("Colab AI gagal menyusun konfigurasi topik (themes kosong). Jalankan ulang cell ini.")
    topic_config["_topic"] = TOPIC
    with open(TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(topic_config, f, ensure_ascii=False, indent=2)
    print(f"[INFO] Konfigurasi topik dari AI disimpan ke {TOPIC_CONFIG_PATH}")

THEME_DESCRIPTIONS = {**topic_config["themes"], "lainnya": "topik di luar kategori di atas"}
THEMES = list(THEME_DESCRIPTIONS)
RELEVANT_THEMES = [t for t in THEMES if t != "lainnya"]
KW_RELEVAN = [k.lower() for k in topic_config.get("keywords", [])]
ENTITY_ALIASES = {k.lower(): v for k, v in topic_config.get("aliases", {}).items()}
EXTRA_MEDIA_ACCOUNTS = {a.lower().lstrip("@") for a in topic_config.get("media_accounts", [])}

# Baris tanpa satu pun keyword langsung "lainnya" (hemat AI + GLiNER). Tanpa keyword = semua baris ke AI.
RE_RELEVAN = re.compile(
    r"(?<!\w)(?:" + "|".join(map(re.escape, sorted(set(KW_RELEVAN), key=len, reverse=True))) + r")(?!\w)"
) if KW_RELEVAN else None


# BUKAN canonical(): nama itu sudah dipakai di Konfigurasi untuk key akun. Kalau ditimpa, Step 2 crash
# ("unhashable type: 'list'") karena canonical('@budi') jadi list huruf.
def canonical_entities(names):
    """Nama entitas lain -> nama baku (ENTITY_ALIASES), sisanya apa adanya."""
    return [ENTITY_ALIASES.get(n.strip().lower(), n.strip()) for n in names]


print(json.dumps(topic_config, ensure_ascii=False, indent=2))

[INFO] Konfigurasi topik dibaca dari /content/drive/MyDrive/topic_config_prabowo_vladivostok.json
{
  "themes": {
    "agenda_pertemuan": "Masuk: Membahas detail teknis kunjungan, agenda pertemuan, dan momen-momen selama acara berlangsung. Contoh frasa: tiba di vladivostok, upacara penyambutan, makan malam kenegaraan, delegasi indonesia, jadwal padat, sesi foto bersama, prabowo disambut putin, konferensi pers, pernyataan bersama, agenda bilateral, perhelatan penting, momen bersejarah, persiapan matang, protokoler, setibanya di. Bukan: Diskusi tentang hasil atau dampak kebijakan yang dibahas (masuk kerja_sama_bilateral) atau reaksi politik (masuk dampak_geopolitik).",
    "kerja_sama_bilateral": "Masuk: Fokus pada isu-isu substantif yang dibahas dan potensi/hasil kerja sama antara Indonesia dan Rusia di berbagai sektor. Contoh frasa: kerja sama ekonomi, investasi pertahanan, perdagangan bilateral, energi nuklir, isu keamanan, forum bisnis, transfer teknologi, ekspor impor, perjanjian da

# Step 1 — Load Data dari Google Sheets

Output: tabel mentah + daftar kolom, buat konfirmasi data ke-load dengan benar.


In [18]:
def xlsx_url(link):
    """Link Google Sheets/Drive -> url download .xlsx. Selain itu (path lokal) dipakai apa adanya."""
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_raw_data(link):
    try:
        df = pd.read_excel(xlsx_url(link), header=1)
    except ValueError as e:
        raise ValueError(f"Gagal baca {link}. Pastikan akses file 'Anyone with the link'.") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]   # KeyError menyebut kolom yang hilang


# Duplikat post lintas file dibuang di Step 3 (dedup_key).
df_raw = pd.concat([load_raw_data(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)],
                   ignore_index=True)
print(f"Data berhasil dimuat: {df_raw.shape}")
display(df_raw.head())

Data berhasil dimuat: (66891, 13)


,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited,source_file
0,1,mention,"Netflix Rilis Skenario Sang Jenderal, Film Dokumenter tentang Kasus Ferdy Sambo","rmol news logo Film dokumenter Skenario Sang Jenderal tayang di Netflix dan mengangkat kembali kasus pembunuhan Brigadir Polisi Nofriansyah Yosua Hutabarat atau Brigadir J yang melibatkan mantan petinggi Polri, Ferdy Sambo. Film dokumenter Skenario Sang Jenderal diproduksi oleh Bleach House Pictures bekerja sama dengan Tempo Media.\n\nDokumenter tersebut mengulas rangkaian peristiwa yang terjadi sejak kematian Brigadir J pada Juli 2022 hingga proses hukum yang menyusul setelahnya. Disutradarai Rob Sixsmith, Skenario Sang Jenderal memiliki durasi 1 jam 29 menit.\n\nFilm ini menampilkan sejumlah rekaman dan gambar terkait kasus tersebut, termasuk kondisi tempat kejadian perkara, proses persidangan, serta keluarga Brigadir J.\n\nFilm dokumenter Skenario Sang Jenderal dibuka dengan narasi mengenai berbagai informasi yang beredar setelah Brigadir J ditemukan tewas. Selain rekaman dari lokasi kejadian, dokumenter ini menghadirkan proses persidangan sejumlah terdakwa.\n\nKesaksian dan keterangan dari sejumlah narasumber juga digunakan untuk membangun kronologi kasus yang sempat menjadi perhatian luas di Indonesia. Film tersebut turut memperlihatkan duka keluarga Brigadir J setelah kematian Yosua.\n\nYosua ditemukan tewas di rumah dinas Ferdy Sambo di kompleks Polri, Duren Tiga, Jakarta Selatan. Pada awal pengungkapan kasus, muncul narasi bahwa Brigadir J tewas dalam baku tembak dengan Bharada Richard Eliezer atau Bharada E.\n\nPeristiwa tersebut dikaitkan dengan dugaan pelecehan seksual terhadap Putri Candrawathi, istri Ferdy Sambo. Seiring proses penyelidikan, narasi awal tersebut kemudian berubah.\n\nKapolri Jenderal Listyo Sigit Prabowo menyatakan bahwa tidak terjadi baku tembak seperti yang sebelumnya disampaikan. Penyidikan selanjutnya mengungkap keterlibatan Ferdy Sambo dalam pembunuhan Brigadir J.\n\nSambo dinyatakan terbukti memerintahkan Bharada E menembak Yosua dan melakukan rekayasa tempat kejadian perkara untuk membangun narasi seolah-olah terjadi baku tembak. Kasus tersebut juga berkembang ke dugaan perusakan dan penghilangan barang bukti, termasuk rekaman kamera CCTV di sekitar lokasi kejadian.\n\nPerkara kemudian mencakup pembunuhan berencana serta obstruction of justice atau tindakan menghalangi proses hukum.\n\nLima orang menjadi terdakwa dalam perkara pembunuhan berencana Brigadir J, yakni Ferdy Sambo, Putri Candrawathi, Richard Eliezer, Ricky Rizal, dan Kuat Ma&#39;ruf. Pengadilan Negeri Jakarta Selatan pada Februari 2023 menjatuhkan hukuman mati kepada Ferdy Sambo.\n\nPutusan tersebut kemudian berubah menjadi hukuman penjara seumur hidup pada tingkat banding. Putri Candrawathi dijatuhi hukuman 20 tahun penjara.\n\nSementara itu, Richard Eliezer atau Bharada E mendapatkan hukuman lebih ringan setelah statusnya sebagai justice collaborator menjadi salah satu pertimbangan dalam proses hukum.\n\nKasus pembunuhan Brigadir J menjadi salah satu perkara yang mendapat perhatian nasional karena melibatkan pejabat tinggi kepolisian. Perkara tersebut juga memunculkan pembahasan mengenai penyalahgunaan kekuasaan, proses penyidikan, serta penanganan barang bukti dalam institusi penegak hukum.&quot;https://dashboard.rmol.id/assets/images/logo/10441704062019_akhir.png&quot;&gt;\n\n",2026-09-22 15:45:00,https://rmol.id/hiburan/read/2026/09/22/722557/netflix-rilis-skenario-sang-jenderal-film-dokumenter-tentang-kasus-ferdy-sambo,News,Negative,rmol.id,0,0,0,file_1
1,2,rt,NaN,"RT Negara hancur, Aturan baru OJK: Danantara, BI, kementerian keuangan, boleh punya saham BEI 😂 Heiii antek-antek asinggg, kabur lah dari investasi Indonesia, regulator dan pengawas ikut main, siap-siap aturannya bakal diobok-obok sesuka prabowo 😂",2026-09-22 15:32:35,https://twitter.com/web/statuses/2102315108575965632,Twitter,Negative,@iwantjiminp,5030,0,0,file_1
2,

# Step 2 - Filter akun media (satu-satunya definisi + eksekusi)


In [33]:
# True = akun media (outlet berita, kanal TV/YouTube) ikut dianalisis dan bisa jadi KOL; tetap diberi tanda is_media.
# False = akun media dibuang di sini, seperti versi sebelumnya (hanya akun non-media).
INCLUDE_MEDIA = True

NUM_COLS = [COL_FOLLOWERS, COL_RETWEETED, COL_FAVOURITED]
df_raw[NUM_COLS] = df_raw[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)

# Author di YouTube/Facebook berupa nama tampilan (ada spasi), tidak dikenali filter username di library.
MEDIA_NAME_WORDS = {"tv", "berita", "news", "info", "terkini", "update", "channel", "official",
                    "media", "portal", "redaksi", "production", "productions"}


def is_media_name(author):
    """'TV Rakyat | Seputar Info Terkini', 'KOMPASTV JAWA BARAT' -> True. Username tanpa spasi tidak dicek di sini."""
    author = str(author).strip()
    words = re.findall(r"[a-z]+", author.lower())
    return " " in author and any(w in MEDIA_NAME_WORDS or w.endswith("tv") for w in words)


assert is_media_name("TV Rakyat | Seputar Info Terkini") and is_media_name("KOMPASTV JAWA BARAT")
assert not is_media_name("Ahmad Dedi") and not is_media_name("@newsfan")

by_account = pd.Series([is_media_account(a, p, extra_accounts=EXTRA_MEDIA_ACCOUNTS)
                        for a, p in zip(df_raw[COL_AUTHOR_ID], df_raw[COL_MEDIA])], index=df_raw.index)
by_name = df_raw[COL_AUTHOR_ID].map(is_media_name)
is_media = by_account | by_name

# merged media: outlet yang ketahuan media lewat SATU ejaan (mis. www.kompastv.com di News) ikut ditandai media
# di ejaan lainnya (@KOMPASTVCOM di Twitter). MEDIA_IDS dipakai lagi di Step 5B untuk node jaringan;
# kolom author_key (key canonical) dipakai lagi di Step 5 dan 5B.
df_raw["author_key"] = df_raw[COL_AUTHOR_ID].map(canonical)
MEDIA_IDS = set(df_raw.loc[is_media, "author_key"].dropna()) | {canonical(a) for a in EXTRA_MEDIA_ACCOUNTS}
by_merge = df_raw["author_key"].isin(MEDIA_IDS) & ~is_media
is_media |= by_merge
df_raw["is_media"] = is_media

print(f"[INFO] {is_media.sum()} baris media terdeteksi ({(by_name & ~by_account).sum()} dari nama kanal, "
      f"{by_merge.sum()} dari ejaan lain outlet yang sama), {(~is_media).sum()} baris non-media")
print("[INFO] Contoh kanal media dari nama:", df_raw.loc[by_name & ~by_account, COL_AUTHOR_ID].unique()[:15].tolist())
print("[INFO] Contoh ejaan media dari merge:", df_raw.loc[by_merge, COL_AUTHOR_ID].unique()[:15].tolist())
display(df_raw.assign(is_media=is_media)[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
        .drop_duplicates(subset=COL_AUTHOR_ID).head(20))

if INCLUDE_MEDIA:
    print(f"[INFO] INCLUDE_MEDIA = True: {is_media.sum()} baris media IKUT dianalisis. "
          "Artikel berita panjang & banyak -> Step 4D lebih lama dan lebih boros kuota Colab AI.")
else:
    df_raw = df_raw[~is_media].reset_index(drop=True)
    print("[INFO] INCLUDE_MEDIA = False: baris media dibuang.")
print(f"[INFO] {df_raw['author_key'].isna().sum()} baris tanpa Author: tetap dipakai untuk tema, tapi tidak ikut ranking KOL")

[INFO] 0 baris media terdeteksi (0 dari nama kanal, 0 dari ejaan lain outlet yang sama), 37333 baris non-media
[INFO] Contoh kanal media dari nama: []
[INFO] Contoh ejaan media dari merge: []


,Author,Media,is_media
0,@iwantjiminp,Twitter,False
1,@wanzhouvers,Twitter,False
2,@moodmasyarakat,Twitter,False
3,@TTariuu,Twitter,False
4,@haowbin,Twitter,False
5,@asheenant,Twitter,False
6,@urmynght,Twitter,False
7,@nong_METAA,Twitter,False
8,@yarawa_n_wo,Twitter,False
9,@WeKa_Ronin,Twitter,False


[INFO] INCLUDE_MEDIA = True: 0 baris media IKUT dianalisis. Artikel berita panjang & banyak -> Step 4D lebih lama dan lebih boros kuota Colab AI.
[INFO] 0 baris tanpa Author: tetap dipakai untuk tema, tapi tidak ikut ranking KOL


# Step 3 — Cleaning text (jangan diubah)

In [34]:
# Export ini menulis retweet sebagai "RT <teks>" (sering TANPA @user) dan balasan sebagai "... [RE username]".
RE_RT_PREFIX = re.compile(r"^\s*RT\b\s*(?:@\w+\s*:\s*)?")      # ponytail: "RT" huruf besar di awal = retweet
RE_RT = re.compile(r"(?<!\w)RT\s*@(\w+)", re.IGNORECASE)        # akun yang di-retweet, kalau tertulis
RE_REPLY = re.compile(r"\[RE\s+@?(\w+)\]", re.IGNORECASE)        # akun yang dibalas
RE_ELONGATION = re.compile(r"(.)\1{2,}")
EMOJI_LANG = "id" if "id" in emoji.LANGUAGES else "en"


def strip_noise(text):
    """Buang prefix RT & penanda [RE user], emoji -> kata, 'mantaaap' -> 'mantaap'. Dipakai untuk BERT dan NER."""
    text = RE_REPLY.sub(" ", RE_RT_PREFIX.sub("", text))
    text = emoji.demojize(text, language=EMOJI_LANG).replace("_", " ").replace(":", " ")
    return RE_ELONGATION.sub(r"\1\1", text)


assert strip_noise("RT @abc: halo [RE def]").strip() == "halo"
assert strip_noise("RT halo").strip() == "halo"


def build_raw_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


def dedup_key(texts):
    """Teks -> kunci pembanding duplikat (huruf kecil, tanpa tanda baca). Dipakai lagi di Step 5B."""
    return texts.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")


df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(df_raw[COL_HEADLINE], df_raw[COL_MENTIONS])]
df_raw["is_retweet"] = df_raw["text_raw_combined"].map(lambda t: bool(RE_RT_PREFIX.match(t)))
noise_free = df_raw["text_raw_combined"].map(strip_noise)
df_raw["text_clean"] = noise_free.progress_apply(clean_for_bert)   # clean_for_bert sudah merapikan spasi
df_raw["text_ner"] = noise_free.apply(clean_for_ner)                # GLiNER: case & slang dipertahankan

df_clean = df_raw[df_raw["text_clean"].str.split().str.len() >= 3].copy()
print(f"[INFO] Buang {len(df_raw) - len(df_clean)} baris kosong/terlalu pendek")

before = len(df_clean)
df_clean = (df_clean.assign(dedup_key=dedup_key(df_clean["text_clean"]))
            .sort_values("is_retweet", kind="stable")   # post asli didahulukan: yang tersimpan milik penulis aslinya
            .drop_duplicates(subset=["dedup_key", COL_AUTHOR_ID] if DEDUP_PER_AUTHOR else ["dedup_key"])
            .sort_index()
            .drop(columns="dedup_key").reset_index(drop=True))
print(f"[INFO] {df_raw['is_retweet'].sum()} retweet terdeteksi. Buang {before - len(df_clean)} duplikat "
      f"(per author: {DEDUP_PER_AUTHOR})")
print(f"[OK] {len(df_clean)} baris tersisa setelah cleaning ({df_clean['is_retweet'].sum()} retweet tanpa post asli di data)")

display(df_clean[["text_raw_combined", "text_clean", "text_ner"]].head(5))

  0%|          | 0/37333 [00:00<?, ?it/s]

[INFO] Buang 1709 baris kosong/terlalu pendek
[INFO] 13816 retweet terdeteksi. Buang 14164 duplikat (per author: False)
[OK] 21460 baris tersisa setelah cleaning (936 retweet tanpa post asli di data)


,text_raw_combined,text_clean,text_ner
0,"RT Negara hancur, Aturan baru OJK: Danantara, BI, kementerian keuangan, boleh punya saham BEI 😂 Heiii antek-antek asinggg, kabur lah dari investasi Indonesia, regulator dan pengawas ikut main, siap-siap aturannya bakal diobok-obok sesuka prabowo 😂","Negara hancur, Aturan baru OJK Danantara, BI, kementerian keuangan, boleh punya saham BEI wajah gembira berurai air mata Heii antek-antek asingg, kabur lah dari investasi Indonesia, regulator dan pengawas ikut main, siap-siap aturannya bakal diobok-obok sesuka prabowo wajah gembira berurai air mata","Negara hancur, Aturan baru OJK Danantara, BI, kementerian keuangan, boleh punya saham BEI wajah gembira berurai air mata Heii antek-antek asingg, kabur lah dari investasi Indonesia, regulator dan pengawas ikut main, siap-siap aturannya bakal diobok-obok sesuka prabowo wajah gembira berurai air mata"
1,"RT ""Prabowo NPD this NPD that.."" . . . .. tapi ntar milih KDM https://t.co/kijt0OfoiU","""Prabowo NPD this NPD that.."" . . . .. tapi nanti milih KDM //t.co/kijt0OfoiU","""Prabowo NPD this NPD that.."" . . . .. tapi ntar milih KDM //t.co/kijt0OfoiU"
2,Pakar Hukum UNPAD Romli Atmasasmita: Presiden Prabowo percaya kepada kinerja Kapolri Listyo Sigit Prabowo🫡 #MoodMasyarakat #Kapolri #Unpad #ListyoSigitPrabowo #FYP https://t.co/iBMKU6uKAN,Pakar Hukum UNPAD Romli Atmasasmita Presiden Prabowo percaya kepada kinerja Kapolri Listyo Sigit Prabowo wajah memberi hormat MoodMasyarakat Kapolri Unpad ListyoSigitPrabowo FYP //t.co/iBMKU6uKAN,Pakar Hukum UNPAD Romli Atmasasmita Presiden Prabowo percaya kepada kinerja Kapolri Listyo Sigit Prabowo wajah memberi hormat Mood Masyarakat Kapolri Unpad Listyo Sigit Prabowo F Y P //t.co/iBMKU6uKAN
3,"emang prabowo sialan, mbg sialan [RE kenhans03]","memang prabowo sialan, mbg sialan","emang prabowo sialan, mbg sialan"
4,RT breaking news gibran lagi di gugat di mk penggugat minta diskualifikasi gibran dari cawapres soalnya dianggap enggak penuhi syarat pendidikan lalu minta mpr gelar sidang buat milih wapres baru dari dua calon usulan prabowo https://t.co/S8DNK7XE5k,breaking news gibran lagi di gugat di mk penggugat minta diskualifikasi gibran dari cawapres soalnya dianggap tidak penuhi syarat pendidikan lalu minta mpr gelar sidang buat milih wapres baru dari dua calon usulan prabowo //t.co/S8DNK7XE5k,breaking news gibran lagi di gugat di mk penggugat minta diskualifikasi gibran dari cawapres soalnya dianggap enggak penuhi syarat pendidikan lalu minta mpr gelar sidang buat milih wapres baru dari dua calon usulan prabowo //t.co/S8DNK7XE5k


# Step 4A — Load GLiNER

In [35]:
import torch
from gliner import GLiNER

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gliner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME).to(DEVICE)

# Potongan dihitung dalam token GLiNER (tanda baca = 1 token), bukan kata, supaya tidak ada yang terpotong.
GLINER_CHUNK_TOKENS = gliner_model.config.max_len
GLINER_CHUNK_OVERLAP = 20   # token yang diulang di awal potongan berikutnya, supaya entitas di batas tidak terbelah

print(f"GLiNER siap di {DEVICE} | potongan {GLINER_CHUNK_TOKENS} token")
if DEVICE == "cpu":
    print("\n⚠️⚠️⚠️ GLiNER jalan di CPU: Step 4D bisa BERJAM-JAM.\n"
          "Ganti ke GPU: Runtime -> Change runtime type -> T4 GPU -> Save,\n"
          "lalu mulai lagi dari Cell ① (ganti runtime = mesin baru, paket harus diinstall ulang).")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

GLiNER siap di cuda | potongan 384 token


# Step 4B — GLiNER extract entitas mentah

In [36]:
RE_GLINER_TOKEN = re.compile(r"\w+(?:[-_]\w+)*|\S")   # sama dengan WhitespaceTokenSplitter bawaan GLiNER


def gliner_chunks(text):
    """Potong teks per GLINER_CHUNK_TOKENS token (tumpang GLINER_CHUNK_OVERLAP), teks asli tetap utuh."""
    toks = list(RE_GLINER_TOKEN.finditer(text))
    step = GLINER_CHUNK_TOKENS - GLINER_CHUNK_OVERLAP
    return [text[toks[i].start():toks[min(i + GLINER_CHUNK_TOKENS, len(toks)) - 1].end()]
            for i in range(0, max(len(toks) - GLINER_CHUNK_OVERLAP, 1), step)]


def gliner_extract_entities(text):
    """Return (entities per label dalam bentuk persis seperti di teks, confidence terendah)."""
    result = {lab: [] for lab in GLINER_LABELS}
    scores = []
    for chunk in gliner_chunks(text):
        for e in gliner_model.predict_entities(chunk, list(GLINER_PROMPT_LABELS), threshold=GLINER_THRESHOLD):
            lab = GLINER_PROMPT_LABELS.get(e["label"])   # "person" -> "tokoh", dst.
            if lab and e["text"] not in result[lab]:
                result[lab].append(e["text"])
                scores.append(e["score"])
    return result, min(scores, default=None)

# Step 4C — Gemini: tema + validasi NER (build_validation_prompt, entities_dict_to_spans)

In [37]:
THEME_LIST_STR = "\n".join(f"- {t}: {d}" for t, d in THEME_DESCRIPTIONS.items())


def build_validation_prompt(batch_items):
    """batch_items: list of (text, gliner_entities_dict)"""
    joined = "\n".join(
        f'{i}. Teks: "{text}"\n'
        f'   Prediksi GLiNER -> lokasi: {ents["lokasi"]}, instansi: {ents["instansi"]}, tokoh: {ents["tokoh"]}'
        for i, (text, ents) in enumerate(batch_items, 1)
    )
    return f"""Kamu classifier tema dan validator NER untuk cuitan bahasa Indonesia tentang: {TOPIC}.

Daftar tema:
{THEME_LIST_STR}

Untuk setiap item, tentukan tema dan periksa prediksi GLiNER: pindahkan entitas yang salah
kategori, tambahkan yang terlewat, hapus yang bukan entitas asli.
- Tulis entitas persis seperti di teks (ejaan, singkatan, huruf besar-kecil). Jangan dinormalisasi atau dilengkapi.
- Jabatan tanpa nama bukan tokoh.
- Tema "lainnya" WAJIB dipakai kalau cuitan tidak membahas topik di atas secara langsung
  (misal hanya menyebut Rusia, Putin, BRICS, geopolitik, atau Prabowo untuk peristiwa lain).

Item:
{joined}

Jawab HANYA JSON array berisi tepat {len(batch_items)} objek, masing-masing dengan field:
index (nomor item), primary_theme (salah satu nama tema di atas), confidence (0-1),
lokasi, instansi, tokoh (list string)."""


def entities_dict_to_spans(text, entities):
    """{"lokasi": [surface,...], ...} -> ([[start, end, label], ...], [nama tidak ketemu]).
    Semua kemunculan, case-insensitive, kata utuh. Nama panjang duluan."""
    pairs = sorted({(n.strip(), lab) for lab, names in entities.items() for n in names if n and n.strip()},
                   key=lambda p: len(p[0]), reverse=True)
    spans, missing = [], []
    for name, label in pairs:
        matches = list(re.finditer(r"(?<!\w)" + re.escape(name) + r"(?!\w)", text, re.IGNORECASE))
        if not matches:
            missing.append(name)
        for m in matches:
            if not any(m.start() < e and s < m.end() for s, e, _ in spans):
                spans.append([m.start(), m.end(), label])
    return sorted(spans), missing

# Step 4D — Eksekusi

In [38]:
# Estimasi waktu ~= (baris / BATCH_SIZE_CLASSIFY) x (detik per batch + SLEEP_BETWEEN_CALLS) / AI_WORKERS
AI_WORKERS = 1       # request AI paralel. 2 sudah kena rate limit di Colab AI; naikkan hanya kalau kuotamu besar.
MAX_AI_ROWS = None   # None = semua baris. Angka = sampel acak (ranking Step 5 jadi dari sampel).
CHECKPOINT_EVERY = 10   # batch

# sidik jari run ini: checkpoint dari topik/tema/data/label GLiNER yang berbeda tidak boleh dilanjutkan
RUN_FINGERPRINT = {
    "topic": TOPIC,
    "themes": THEMES,
    "gliner_labels": GLINER_PROMPT_LABELS,
    "n_rows": len(df_clean),
    "data_hash": int(pd.util.hash_pandas_object(df_clean["text_clean"], index=False).sum()),
}


def save_checkpoint(next_b):
    pd.to_pickle({"fingerprint": RUN_FINGERPRINT, "df": df_clean, "gliner": gliner_results,
                  "review_idx": review_idx, "next_b": next_b, "ner": ner_training_data,
                  "tema": tema_training_data, "n_skip": n_skip_missing},
                 CHECKPOINT_PATH)


def load_checkpoint():
    """Checkpoint milik run ini, atau None. Yang tidak bisa dibaca (beda versi pandas, file rusak)
    atau milik data/topik lain dihapus supaya run mulai dari awal, bukan crash."""
    if not os.path.exists(CHECKPOINT_PATH):
        return None
    try:
        ckpt = pd.read_pickle(CHECKPOINT_PATH)
    except Exception as e:
        print(f"[INFO] Checkpoint lama tidak bisa dibaca ({type(e).__name__}) -> dihapus, mulai dari awal")
        os.remove(CHECKPOINT_PATH)
        return None
    if ckpt.get("fingerprint") != RUN_FINGERPRINT:
        print("[INFO] Checkpoint lama berasal dari data/topik/label yang berbeda -> dihapus, mulai dari awal")
        os.remove(CHECKPOINT_PATH)
        return None
    return ckpt


def validate_batch(b):
    """Satu request AI (jalan di thread). Return (index df, hasil per item atau None)."""
    idx_batch = review_idx[b * BATCH_SIZE_CLASSIFY:(b + 1) * BATCH_SIZE_CLASSIFY]
    items = [(df_clean.at[i, "text_ner"], gliner_results[i]["entities"]) for i in idx_batch]
    # hasil dicocokkan lewat field 'index', BUKAN posisi
    return idx_batch, ask_json(build_validation_prompt(items),
                               parse=lambda p: order_by_index(p, len(items)), retry=6)


def set_entities(idx, ents, source):
    for lab in GLINER_LABELS:
        df_clean.at[idx, f"entities_{lab}"] = canonical_entities(ents[lab])
    df_clean.at[idx, "ner_source"] = source


ckpt = load_checkpoint()
if ckpt is not None:
    df_clean, gliner_results, review_idx, start_b = ckpt["df"], ckpt["gliner"], ckpt["review_idx"], ckpt["next_b"]
    ner_training_data, tema_training_data, n_skip_missing = ckpt["ner"], ckpt["tema"], ckpt["n_skip"]
    print(f"[RESUME] Lanjut dari batch {start_b} (checkpoint {CHECKPOINT_PATH})")
else:
    df_clean[["theme", "theme_confidence", "ner_source"]] = None
    df_clean[["ner_failed", "from_prefilter", "gliner_corrected"]] = False
    for lab in GLINER_LABELS:
        df_clean[f"entities_{lab}"] = [[] for _ in range(len(df_clean))]

    # tahap 1: keyword prefilter -> tanpa satu pun keyword relevan langsung "lainnya"
    if RE_RELEVAN:
        text_lower = df_clean["text_clean"].str.lower()
        irrelevant = ~text_lower.str.contains(RE_RELEVAN)
        df_clean.loc[irrelevant, ["theme", "theme_confidence", "from_prefilter"]] = ["lainnya", 1.0, True]

        kw_hits = Counter(k for t in text_lower[~irrelevant] for k in set(RE_RELEVAN.findall(t)))
        print("[INFO] Keyword yang paling banyak meloloskan baris:", kw_hits.most_common(10))
        if (~irrelevant).mean() > 0.5:
            print(f"[WARN] {(~irrelevant).mean():.0%} baris lolos prefilter: ada keyword yang terlalu umum (lihat di atas). "
                  "Proses jadi lama & boros kuota. Centang REGENERATE_TOPIC_CONFIG di Cell ② lalu jalankan ulang dari sana.")
    review_idx = df_clean.index[df_clean["theme"].isna()].tolist()
    print(f"[INFO] Prefilter: {df_clean['from_prefilter'].sum()} baris 'lainnya', {len(review_idx)} lanjut ke GLiNER + AI")
    if MAX_AI_ROWS and len(review_idx) > MAX_AI_ROWS:
        review_idx = sorted(random.sample(review_idx, MAX_AI_ROWS))
        print(f"[WARN] Sampling: hanya {MAX_AI_ROWS} baris diproses, sisanya tanpa tema (ranking dari sampel)")

    # tahap 2: GLiNER di baris yang lolos prefilter
    gliner_results = {}
    for idx in tqdm(review_idx, desc="GLiNER NER"):
        ents, min_conf = gliner_extract_entities(df_clean.at[idx, "text_ner"])
        gliner_results[idx] = {"entities": ents, "min_conf": min_conf}
        set_entities(idx, ents, "gliner")

    ner_training_data, tema_training_data = [], []
    n_skip_missing = 0
    start_b = 0
    save_checkpoint(0)   # GLiNER juga lama, jangan diulang kalau putus

# tahap 3: AI tentukan tema + validasi NER, AI_WORKERS batch sekaligus
n_batches = -(-len(review_idx) // BATCH_SIZE_CLASSIFY)
print(f"[INFO] Sisa {n_batches - start_b} batch, {AI_WORKERS} paralel")
pbar = tqdm(total=n_batches, initial=start_b, desc="Validate NER+Tema")
finished = True

with ThreadPoolExecutor(AI_WORKERS) as pool:
    for b0 in range(start_b, n_batches, AI_WORKERS):
        next_b = min(b0 + AI_WORKERS, n_batches)
        try:
            done = list(pool.map(validate_batch, range(b0, next_b)))
        except Exception as e:
            if not is_quota_error(e):
                raise
            save_checkpoint(b0)   # batch di putaran ini diulang saat lanjut
            print(f"[STOP] Kuota habis / rate limit di batch {b0}/{n_batches}. "
                  "Tunggu beberapa menit, lalu jalankan ulang cell ini (otomatis lanjut dari batch ini).")
            finished = False
            break

        for idx_batch, results in done:
            if results is None:
                # tema tetap None (terhitung belum diproses), entity dari GLiNER, tidak masuk training
                print("[ERROR] Batch validasi gagal total. Entity tetap dari GLiNER, baris TIDAK masuk training.")
                df_clean.loc[idx_batch, "ner_failed"] = True
                continue

            for r, df_idx in zip(results, idx_batch):
                theme = r.get("primary_theme")
                theme = theme if theme in THEMES else "lainnya"
                conf = float(r.get("confidence", 0.5))
                df_clean.at[df_idx, "theme"] = theme
                df_clean.at[df_idx, "theme_confidence"] = conf
                tema_training_data.append({"text": df_clean.at[df_idx, "text_clean"], "label": theme,
                                           "confidence": conf, "label_source": "gemini"})

                ents = {lab: [x for x in (r.get(lab) or []) if isinstance(x, str)] for lab in GLINER_LABELS}
                set_entities(df_idx, ents, "gemini")

                gl = gliner_results[df_idx]["entities"]
                df_clean.at[df_idx, "gliner_corrected"] = any(
                    sorted(x.lower() for x in gl[lab]) != sorted(x.lower() for x in ents[lab]) for lab in GLINER_LABELS)

                text_ner = df_clean.at[df_idx, "text_ner"]
                spans, missing = entities_dict_to_spans(text_ner, ents)
                if missing:
                    n_skip_missing += 1   # label parsial = racun training, jangan dipakai
                    continue
                ner_training_data.append({
                    "text": text_ner, "entities": spans, "label_source": "gemini",
                    "review_reason": "tema_ambigu", "model_version": GLINER_MODEL_NAME,
                    "last_confidence": gliner_results[df_idx]["min_conf"],
                })

        pbar.update(next_b - b0)
        if next_b // CHECKPOINT_EVERY != b0 // CHECKPOINT_EVERY:
            save_checkpoint(next_b)
        time.sleep(SLEEP_BETWEEN_CALLS)
pbar.close()

if finished:
    # training data ditulis sekali di sini (bukan tiap resume, supaya tidak dobel)
    save_pending("ner", ner_training_data)
    save_pending("tema", tema_training_data)
    os.remove(CHECKPOINT_PATH)

validated = df_clean["ner_source"] == "gemini"
if validated.any():
    rate = df_clean.loc[validated, "gliner_corrected"].mean()
    print(f"\n[INFO] correction_rate GLiNER: {100 * rate:.1f}% dari {validated.sum()} baris tervalidasi")
print(f"[INFO] {n_skip_missing} baris tidak masuk training (entity AI tidak ditemukan di teks)")

n_unprocessed = df_clean["theme"].isna().sum()
if n_unprocessed:
    print(f"[WARN] {n_unprocessed} baris BELUM punya tema (sampling / kuota habis / batch gagal). "
          "Laporan di bawah TIDAK lengkap.")

print("\nDistribusi tema:")
display(df_clean["theme"].value_counts(dropna=False))

print("\nContoh hasil klasifikasi + NER:")
display(df_clean.loc[df_clean["theme"].isin(RELEVANT_THEMES),
        ["text_clean", "theme", "ner_source", "entities_lokasi", "entities_instansi", "entities_tokoh"]].head(10))

df_clean.to_pickle(SAVE_PATH_AFTER_THEME)
print(f"[OK] df_clean tersimpan ke {SAVE_PATH_AFTER_THEME}. Total API call: {api_call_count}")

[INFO] Keyword yang paling banyak meloloskan baris: [('rusia', 124), ('putin', 52), ('vladimir putin', 13), ('vladivostok', 12), ('moskow', 4), ('prabowo ke rusia', 4), ('prabowo putin', 3), ('federasi rusia', 2), ('prabowo rusia', 2), ('forum ekonomi timur', 1)]
[INFO] Prefilter: 21284 baris 'lainnya', 176 lanjut ke GLiNER + AI


GLiNER NER:   0%|          | 0/176 [00:00<?, ?it/s]

[INFO] Sisa 5 batch, 1 paralel


Validate NER+Tema:   0%|          | 0/5 [00:00<?, ?it/s]

[WARN] AI gagal (1/6): Expecting ',' delimiter: line 697 column 12 (char 12268) -> tunggu 5s
[pending] ner: 166 baris ditulis ke /content/drive/MyDrive/pending_ner_data.jsonl
[pending] tema: 176 baris ditulis ke /content/drive/MyDrive/pending_tema_data.jsonl

[INFO] correction_rate GLiNER: 97.7% dari 176 baris tervalidasi
[INFO] 10 baris tidak masuk training (entity AI tidak ditemukan di teks)

Distribusi tema:


,count
theme,
lainnya,21371
dampak_geopolitik,48
kerja_sama_bilateral,28
agenda_pertemuan,13



Contoh hasil klasifikasi + NER:


,text_clean,theme,ner_source,entities_lokasi,entities_instansi,entities_tokoh
303,"Comparison Statement Analysis Presiden RI di EEF(Eastern Economic Forum) 3 Sept 2026, Vladivostok Russia. Yang dimuat di The Star dan The Straits Time. //t.co/b5ATZhtHjE",agenda_pertemuan,gemini,"[Vladivostok, Russia]","[EEF, Eastern Economic Forum, The Star, The Straits Time]",[]
710,"Bahlil Lahadalia on TikTok Hari ini saya mendampingi Bapak Presiden Prabowo bertemu dengan Presiden Federasi Rusia Vladimir Putin di Vladivostok dalam rangkaian Forum Ekonomi Timur (EEF) ke-11. Rusia telah menjadi mitra penting bagi Indonesia selama lebih dari 5 dasawarsa, termasuk dalam pengembangan kerja sama di sektor energi, perdagangan, teknologi, dan berbagai bidang strategis lainnya. Sebagai dua negara Pasifik, Indonesia dan Rusia memiliki peluang besar untuk terus memperkuat konektivitas dan kerja sama yang memberikan manfaat bagi kedua negara.",kerja_sama_bilateral,gemini,"[Vladivostok, Rusia, Indonesia, Federasi Rusia]","[TikTok, Forum Ekonomi Timur, EEF]","[Bahlil Lahadalia, Prabowo Subianto, Vladimir Putin]"
2862,"TRUMP MERADANG, PRABOWO BORONG ALUTSISTA BUATAN RUSIA! Indonesia 10x lipat Lebih Kuat. TRUMP MERADANG, PRABOWO BORONG ALUTSISTA BUATAN RUSIA! Indonesia 10x lipat Lebih Kuat. Indonesia kembali jadi sorotan setelah Presiden Prabowo Subianto memperkuat modernisasi militer dengan deretan alutsista strategis. Reuters melaporkan India akan memasok rudal jelajah supersonik BrahMos dan rudal udara-ke-udara Astra ke Indonesia dalam kesepakatan sekitar US$630 juta. BrahMos dikembangkan bersama India dan Rusia, sementara Astra disebut bisa diintegrasikan dengan jet Sukhoi buatan Rusia yang digunakan TNI AU. dart Benarkah pembelian BrahMos membuat kekuatan militer Indonesia naik level? dart Mengapa rudal supersonik India-Rusia ini dianggap penting untuk pertahanan laut Indonesia? dart Apakah modernisasi alutsista Prabowo membuat Amerika, China, dan negara kawasan mulai memperhitungkan Indonesia? dart Dan apakah kombinasi Rafale, Sukhoi, BrahMos, dan Astra bisa mengubah peta kekuatan TNI di Indo-Pasifik? == Subscribe disini ya //goo.gl/qHsfPw == Follow kami juga disini menunjuk ke arah kanan Facebook //ww.facebook.com/autopopuler/ menunjuk ke arah kanan Instagram //ww.instagram.com/autopopuler.official/?hl=id -- For Business adv.autopopuler .com",kerja_sama_bilateral,gemini,"[Rusia, Indonesia, India, Rusia, Amerika, China, Indo-Pasifik]","[Reuters, Instagram, TNI AU, TNI, Facebook]","[TRUMP, Prabowo Subianto]"
2959,Tenaga Ahli KSP Curiga Teknologi HAARP AS Penyebab Gunung Erupsi Saat Prabowo Bertemu Putin //t.co/ApwXSRrfRp,agenda_pertemuan,gemini,[AS],"[KSP, HAARP]","[Vladimir Putin, Prabowo Subianto]"
3866,"Momen Dimana Prabowo Ditanya Soal Perjanjian Perdagangan Yang Sudah Dibuatnya, Di Akhir Putin Menyindir Kebijakan Yang Diambil Prabowo //t.co/B8a0ONP2mc",kerja_sama_bilateral,gemini,[],[],"[Prabowo Subianto, Vladimir Putin]"
3880,BRICS2026 BRICSSummit BharatMandapam NarendraModi PMModi Modi VladimirPutin Putin XiJinping Xi LulaDaSilva Lula CyrilRamaphosa MohammedBinSalman MBS MBZ ElSisi Prabowo BRICSLeaders BRICSNations DonaldTrump Trump POTUS USPolitics NewWorldOrder Putin //t.co/k1SONr8Mnw,dampak_geopolitik,gemini,[Bharat Mandapam],[B R I C S],"[Narendra Modi, Vladimir Putin, Vladimir Putin, Xi Jinping, Xi, Lula Da Silva, Lula, Cyril Ramaphosa, Mohammed Bin Salman, M B S, M B Z, El Sisi, Prabowo Subianto, Donald Trump, Trump]"
3886,JUST IN BRICS leaders take family photo at the 2026 Summit. bendera tiongkok Xi Jinping bendera rusia Vladimir Putin bendera india Narendra Modi bendera indonesia Prabowo Subianto bendera iran Masoud Pezeshkian bendera mesir Abdel Fattah el-Sisi bendera vietnam Le Minh Hung bendera afrika selatan Cyril Ramaphosa bendera malaysia Anwar Ibrahim //t.co/62kyE3ykge,dampak_geopolitik,gemini,"[tiongkok, Rusia, india, indonesia, iran, mesir, vietnam, afrika selatan, malaysia]",[BRICS],"[Xi Jinping, Vladimir Putin, Narendra Modi, 

[OK] df_clean tersimpan ke /content/drive/MyDrive/df_clean_after_theme_prabowo_vladivostok.pkl. Total API call: 37


# Step 5 - Ranking Top Author per Tema

Hanya akun non-media (sudah difilter Step 2). Skor = post count + engagement + followers.

In [39]:
processed = df_clean["theme"].notna()
relevant = processed & (df_clean["theme"] != "lainnya")
confident = df_clean["theme_confidence"].astype(float) >= RANKING_MIN_CONFIDENCE

df_topic = df_clean[relevant & confident].copy()
print(f"[INFO] {len(df_topic)} baris ikut ranking, "
      f"{(relevant & ~confident).sum()} baris dibuang karena tema ragu (< {RANKING_MIN_CONFIDENCE})")
if (~processed).sum():
    print(f"[WARN] {(~processed).sum()} baris belum diproses. Ranking PARSIAL.")

df_topic["engagement"] = df_topic[COL_FAVOURITED] + df_topic[COL_RETWEETED]
# dihitung ulang di sini (bukan dari Step 2) supaya tetap ada walau df_clean berasal dari checkpoint lama
df_topic["author_key"] = df_topic[COL_AUTHOR_ID].map(canonical)


def top_spelling(frame):
    """author_key -> ejaan Author yang paling sering muncul (label yang enak dibaca di laporan & Gephi)."""
    return (frame.groupby(["author_key", COL_AUTHOR_ID]).size().sort_values(ascending=False)
            .reset_index().drop_duplicates("author_key").set_index("author_key")[COL_AUTHOR_ID])


# retweet bukan karya author-nya (dan kolom Retweeted-nya milik tweet asli) -> tidak dihitung ke volume.
# pengaruhnya masuk ke penulis asli lewat jaringan di Step 5B.
df_original = df_topic[~df_topic["is_retweet"]]
print(f"[INFO] {len(df_topic) - len(df_original)} retweet tidak dihitung ke skor volume (masuk ke jaringan)")

# dikelompokkan per AKUN (author_key), bukan per ejaan: 'prabowo.subianto' (IG) + '@Prabowo_Subianto' = 1 akun
agg = (df_original.groupby(["theme", "author_key"])
       .agg(jumlah_post=("engagement", "size"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"))
       .reset_index()
       .rename(columns={"author_key": "Id"}))
agg["author_id"] = agg["Id"].map(top_spelling(df_original))
# tiap metrik dinormalisasi ke max-nya dalam tema yang sama
metrics = agg[["jumlah_post", "total_engagement", "followers"]].astype(float)
norm = metrics / metrics.groupby(agg["theme"]).transform("max").replace(0, 1)
agg["volume_score"] = norm @ [WEIGHT_POST_COUNT, WEIGHT_ENGAGEMENT, WEIGHT_FOLLOWERS]
print(f"[INFO] Skor volume dihitung untuk {len(agg)} pasangan akun-tema "
      f"({df_original[COL_AUTHOR_ID].nunique()} ejaan Author -> {df_original['author_key'].nunique()} akun)")

[INFO] 89 baris ikut ranking, 0 baris dibuang karena tema ragu (< 0.6)
[INFO] 11 retweet tidak dihitung ke skor volume (masuk ke jaringan)
[INFO] Skor volume dihitung untuk 70 pasangan akun-tema (66 ejaan Author -> 67 akun)


# Step 5B - Nodes & Edges + KOL gabungan (volume + jaringan)

In [40]:
import numpy as np

HANDLE = re.compile(r"(?<![\w.])@([\w.]*\w)")   # dari extractor: handle bertitik (IG) ikut, e-mail tidak


def is_media_id(key):
    """Key akun media: dikenali lewat ejaan mana pun di Step 2 (MEDIA_IDS), atau dari key-nya sendiri."""
    return key in MEDIA_IDS or is_media_account(key, extra_accounts=EXTRA_MEDIA_ACCOUNTS)


def extract_targets(text, retweet_of=None):
    """Satu post -> maksimal satu edge per akun tujuan (key canonical). Prioritas: retweet > reply > mention.
    retweet_of = penulis asli untuk retweet yang tidak menulis @user (dicari lewat teks yang sama)."""
    rt = RE_RT.search(text)
    source = canonical(rt.group(1)) if rt else retweet_of
    targets = {source: "retweet"} if source else {}
    for m in RE_REPLY.findall(text):
        targets.setdefault(canonical(m), "reply")
    for m in HANDLE.findall(text):
        targets.setdefault(canonical(m), "mention")
    return list(targets.items())


def centrality(pair_w):
    """PageRank (graf berarah) + Eigenvector (graf tak berarah, giant component; di luar giant = 0) per akun."""
    G = nx.from_pandas_edgelist(pair_w, "Source", "Target", edge_attr="Weight", create_using=nx.DiGraph)
    pair = np.sort(pair_w[["Source", "Target"]].to_numpy(), axis=1)   # A->B dan B->A dijumlah
    und = pair_w.assign(u=pair[:, 0], v=pair[:, 1]).groupby(["u", "v"], as_index=False)["Weight"].sum()
    U = nx.from_pandas_edgelist(und, "u", "v", edge_attr="Weight")
    giant = U.subgraph(max(nx.connected_components(U), key=len))
    # eigenvector_centrality_numpy butuh >= 3 node (batas ARPACK); 2 node yang saling terhubung = setara
    ev = nx.eigenvector_centrality_numpy(giant, weight="Weight") if len(giant) >= 3 else dict.fromkeys(giant, 1.0)
    out = pd.DataFrame({"pagerank": pd.Series(nx.pagerank(G, weight="Weight"))})
    out["eigenvector"] = pd.Series(ev).reindex(out.index, fill_value=0.0)
    return out.rename_axis("Id").reset_index()


assert extract_targets("RT @Abc: halo @Def @abc. @def, email x@gmail.com") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("RT halo @def", retweet_of="abc") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("setuju banget [RE Abc]") == [("abc", "reply")]
assert extract_targets("cc @prabowo.subianto @Prabowo_Subianto") == [("prabowosubianto", "mention")]
_toy = centrality(pd.DataFrame({"Source": ["a", "b", "c", "x"], "Target": ["b", "c", "a", "y"], "Weight": [1, 1, 1, 5]}))
assert _toy.set_index("Id").loc[["x", "y"], "eigenvector"].eq(0).all()            # di luar giant component
assert abs(_toy["pagerank"].sum() - 1) < 1e-6

# --- merge akun: semua ejaan Author -> satu key (author_key dibuat di Step 2) ---
author_key = df_raw["author_key"]
merges = (df_raw.assign(Author=df_raw[COL_AUTHOR_ID].astype(str))
          .groupby("author_key")["Author"].agg(ejaan=lambda s: sorted(s.unique()), baris="size"))
merges = merges[merges["ejaan"].map(len) > 1].sort_values("baris", ascending=False)
print(f"[INFO] {df_raw[COL_AUTHOR_ID].nunique():,} ejaan Author -> {author_key.nunique():,} akun; "
      f"{len(merges):,} akun menggabungkan 2+ ejaan (cek di bawah; salah gabung -> perbaiki lewat ALIASES)")
display(merges)

# --- edges: dari df_raw (sebelum dedup) supaya setiap retweet terhitung ---
raw_keys = dedup_key(df_raw["text_clean"])
# tema diturunkan dari df_topic -> retweet ikut tema post aslinya, dengan syarat confidence yang sama
theme_by_key = dict(zip(dedup_key(df_topic["text_clean"]), df_topic["theme"]))
raw_theme = raw_keys.map(theme_by_key)
# "RT <teks>" tanpa @user: penulis asli = author post non-retweet dengan teks yang sama
orig_author = (pd.DataFrame({"key": raw_keys, "author": author_key})[~df_raw["is_retweet"]]
               .drop_duplicates("key").set_index("key")["author"])
retweet_of = raw_keys.map(orig_author).where(df_raw["is_retweet"])
print(f"[INFO] {df_raw['is_retweet'].sum()} retweet, {retweet_of.notna().sum()} di antaranya ketemu post aslinya di data")

edge_rows = pd.DataFrame([
    {"Source": author, "Target": target, "interaction": kind, "theme": theme, "text": text}
    for author, raw, text, theme, rt_of in zip(author_key, df_raw["text_raw_combined"],
                                               df_raw["text_clean"], raw_theme, retweet_of)
    if pd.notna(theme) and author is not None   # post tanpa Author tidak punya asal edge
    for target, kind in extract_targets(raw, rt_of if pd.notna(rt_of) else None)
    if target != author
], columns=["Source", "Target", "interaction", "theme", "text"])

# kolom 'Type' sengaja tidak dipakai: di Gephi itu kolom khusus Directed/Undirected
df_edges = edge_rows.groupby(["Source", "Target", "interaction", "theme"]).size().rename("Weight").reset_index()

# teks yang me-retweet X = kata-kata X sendiri -> dipakai Step 6B untuk KOL yang tidak posting di data
rt_posts = (edge_rows[edge_rows["interaction"] == "retweet"]
            .groupby(["Target", "theme"])["text"].apply(lambda s: s.drop_duplicates().tolist()).to_dict())

# --- PageRank + Eigenvector per tema ---
net_parts = [centrality(sub.groupby(["Source", "Target"], as_index=False)["Weight"].sum()).assign(theme=theme)
             for theme, sub in df_edges.groupby("theme")]
df_net = (pd.concat(net_parts, ignore_index=True) if net_parts
          else pd.DataFrame(columns=["Id", "pagerank", "eigenvector", "theme"]))

# --- gabung volume + jaringan ---
followers_by_id = df_raw.assign(Id=author_key).groupby("Id")[COL_FOLLOWERS].max()

kol = agg.merge(df_net, on=["theme", "Id"], how="outer")
# INCLUDE_MEDIA (Step 2) = False -> akun media tetap jadi node jaringan, tapi tidak boleh jadi KOL
if not INCLUDE_MEDIA:
    kol = kol[~kol["Id"].map(is_media_id)]
kol = kol.copy()
kol["is_media"] = kol["Id"].map(is_media_id)
kol["author_id"] = kol["author_id"].fillna(kol["Id"])                   # KOL yang cuma muncul di jaringan
kol["followers"] = kol["followers"].fillna(kol["Id"].map(followers_by_id))
score_cols = ["jumlah_post", "total_engagement", "followers", "volume_score", "pagerank", "eigenvector"]
kol[score_cols] = kol[score_cols].fillna(0)
# dinormalisasi ke max tema SETELAH filter media (kalau INCLUDE_MEDIA = False), supaya skor antar akun sebanding
norm_theme = lambda col: kol[col] / kol.groupby("theme")[col].transform("max").replace(0, 1)
kol["network_score"] = NET_WEIGHT_PAGERANK * norm_theme("pagerank") + NET_WEIGHT_EIGEN * norm_theme("eigenvector")
kol["score"] = KOL_WEIGHT_VOLUME * kol["volume_score"] + KOL_WEIGHT_NETWORK * kol["network_score"]

df_top_authors = (kol.sort_values(["theme", "score"], ascending=[True, False])
                  .groupby("theme").head(TOP_N_AUTHORS_PER_THEME).reset_index(drop=True))
df_top_authors["rank"] = df_top_authors.groupby("theme").cumcount() + 1

# --- nodes (Id unik, format Gephi); PageRank & Eigenvector di sini dihitung atas seluruh jaringan ---
best = kol.sort_values("score").drop_duplicates("Id", keep="last").set_index("Id")
pair_all = df_edges.groupby(["Source", "Target"], as_index=False)["Weight"].sum()
top_ids = set(df_top_authors["Id"])
glob = centrality(pair_all).set_index("Id") if len(pair_all) else pd.DataFrame(columns=["pagerank", "eigenvector"])

df_nodes = pd.DataFrame({"Id": sorted(set(pair_all["Source"]) | set(pair_all["Target"]) | top_ids)})
df_nodes["Label"] = df_nodes["Id"].map(top_spelling(df_raw)).fillna(df_nodes["Id"])   # ejaan asli, bukan key
df_nodes["in_weight"] = df_nodes["Id"].map(pair_all.groupby("Target")["Weight"].sum()).fillna(0).astype(int)
df_nodes["out_weight"] = df_nodes["Id"].map(pair_all.groupby("Source")["Weight"].sum()).fillna(0).astype(int)
df_nodes["pagerank"] = df_nodes["Id"].map(glob["pagerank"]).fillna(0)
df_nodes["eigenvector"] = df_nodes["Id"].map(glob["eigenvector"]).fillna(0)
df_nodes["followers"] = df_nodes["Id"].map(followers_by_id)   # NaN = cuma muncul sebagai target
df_nodes["is_media"] = df_nodes["Id"].map(is_media_id)
df_nodes["kol_theme"] = df_nodes["Id"].map(best["theme"])
df_nodes["kol_score"] = df_nodes["Id"].map(best["score"]).fillna(0)
df_nodes["is_kol"] = df_nodes["Id"].isin(top_ids)

print(f"[OK] {len(df_nodes)} nodes, {len(df_edges)} edges, {len(df_top_authors)} KOL")
print(f"[INFO] Edge per interaksi:\n{df_edges.groupby('interaction')['Weight'].sum()}")

# cek kolom Author = username; kalau tidak, jaringan tidak nyambung ke skor volume
if len(df_edges):
    match_rate = df_edges["Target"].isin(followers_by_id.index).mean()
    print(f"[INFO] {match_rate:.0%} target edge juga muncul sebagai Author di data. "
          "Kalau mendekati 0%, cek apakah kolom Author berisi username (bukan nama tampilan).")

for theme in RELEVANT_THEMES:
    sub = df_top_authors[df_top_authors["theme"] == theme]
    if sub.empty:
        print(f"[INFO] Tidak ada data untuk tema '{theme}'")
        continue
    print(f"\n=== KOL: {theme} ===")
    display(sub[["rank", "author_id", "is_media", "jumlah_post", "total_engagement", "followers",
                 "pagerank", "eigenvector", "volume_score", "network_score", "score"]])

print("\n=== Nodes (urut PageRank) ===")
display(df_nodes.sort_values("pagerank", ascending=False).reset_index(drop=True))
print("\n=== Edges (urut Weight) ===")
display(df_edges.sort_values("Weight", ascending=False).reset_index(drop=True))

[INFO] 17,217 ejaan Author -> 17,105 akun; 65 akun menggabungkan 2+ ejaan (cek di bawah; salah gabung -> perbaiki lewat ALIASES)


,ejaan,baris
author_key,,
,"[....._....., ÃÂ§ÃÂ¤ÃÂÃÂÃÂ§ÃÂ¡, ÃÂ¨ÃÂ§ÃÂÃÂÃÂ³ÃÂ·, ÃÂ²ÃÂÃÂ§ ÃÂ´ÃÂ§ÃÂÃÂÃÂªÃÂ±ÃÂ§, ÃÂºÃÂÃÂÃÂÃÂ, ÃÂÃÂÃÂ¨ÃÂ ÃÂÃÂ§ÃÂ ÃÂÃÂÃÂ§, Ã Â¸ÂÃ Â¸Â§Ã Â¸Â²Ã Â¸Â¡Ã Â¸Â¥Ã Â¸Â±Ã Â¸ÂÃ Â¸ÂÃ Â¸ÂµÃ Â¹ÂÃ Â¹ÂÃ Â¸Â¡Ã Â¹ÂÃ Â¹ÂÃ Â¸ÂÃ Â¸Â´Ã Â¸ÂÃ Â¹ÂÃ Â¸ÂÃ Â¸Â¢Ã Â¸ÂÃ , Ã¡Â´ÂÃÂÃªÂÂ°ÃÂªÃ¡Â´ÂÃÂ´ Ã¡Â´Â¡ÃÂªÃ¡Â´Â ÃÂªÃ¡Â´ÂÃ¡Â´ÂÃ¡Â´ÂÃ¡Â´ÂÃ°ÂÂÂµÃ°ÂÂÂ¸, Ã¡Â´ÂÃÂªÃÂÃÂÃ¡Â´ÂÃÂ´ Ã¡Â´Â ÃÂªÃ¡Â´Â ÃÂªÃ¡Â´ÂÃÂ´Ã¡Â´Â, Ã¢ÂÂ¨, Ã£ÂÂÃ£ÂÂÃ£ÂÂ¢, Ã¥Â¾Â·Ã¨ÂÂ£, Ã§ÂÂ§Ã¦ÂÂÃ°ÂÂÂ®Ã°ÂÂÂ© Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ, Ã°ÂÂÂÃ°ÂÂÂ-Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂ, Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ, Ã°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂ¦Ã°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂ¤ Ã°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂ¬Ã°ÂÂÂ­ | Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¢Ã°ÂÂÂ, Ã°ÂÂÂ©Ã°ÂÂÂ¶Ã°ÂÂÂ¹Ã°ÂÂÂ¾Ã°ÂÂÂ¶ Ã°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¿Ã°ÂÂÂ¶, Ã°ÂÂÂ«Ã°ÂÂÂÃ°ÂÂÂ¿Ã°ÂÂÂ¾, Ã°ÂÂÂÃ°ÂÂÂ®Ã°ÂÂÂ°Ã°ÂÂÂª Ã°ÂÂÂÃ°ÂÂÂ¾Ã°ÂÂÂ½Ã°ÂÂÂ»Ã°ÂÂÂ² Ã°ÂÂÂÃ°ÂÂÂ·Ã°ÂÂÂ°Ã°ÂÂÂ®Ã°ÂÂÂµÃ°ÂÂÂ²Ã°ÂÂ, Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¥Ã°ÂÂÂÃ°ÂÂÂ§Ã°ÂÂÂÃ°ÂÂÂ Ã¢ÂÂÃ°ÂÂÂ¦Ã°ÂÂÂ§Ã°ÂÂÂÃ°ÂÂÂ¥Ã°ÂÂÂÃ°ÂÂÂ¤Ã°ÂÂÂÃ°ÂÂÂ£Ã, Ã°ÂÂÂ»Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ & Ã°ÂÂÂ®Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ, Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ§Ã°ÂÂÂ©Ã°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¨Ã°ÂÂÂÃ°ÂÂÂ¤Ã°ÂÂÂ£Ã°ÂÂÂÃ°ÂÂÂ¡, Ã°ÂÂÂ³Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¢Ã°ÂÂÂ£Ã°ÂÂÂ, Ã°ÂÂÂ½Ã°ÂÂÂÃ°ÂÂÂ Ã°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂÃ°ÂÂÂ¢Ã°ÂÂÂ, Ã°ÂÂÂ, Ã°ÂÂÂ»Ã°ÂÂÂ»Ã°ÂÂÂ», Ã°ÂÂ¤Â , محب الخير , ც𝓵𝒶𝒸𝓀',𝓞𝔀𝓵^, ◟♯ . / kᥱіk᥆ •ᗢ, 大老远啧啧, 멍이, 𝐀𝐬𝐚𝐥𝐁𝐞𝐫𝐬𝐮𝐚, 𝐊𝐢𝐜𝐚𝐮 𝐆𝐚𝐫𝐮𝐝𝐚, 𝐓𝐎𝐊𝐎 𝐊𝐃𝐌, 𝑦𝑎𝑛𝑧 𝑫 𝒏𝒚𝒂𝒘𝒊𝒕, 𝑩𝑨𝑵𝑮 𝑨𝑮𝑼𝑺 𝑪𝑯𝑨𝑵𝑵𝑬𝑳 , 𝓒𝓐𝓟𝓨 & 𝓟𝓪𝓷𝓭𝔂, 𝓗𝓪𝓷𝓷𝓲🦦ྀི , 𝕾𝖊𝖔𝖔_𝖌𝖗𝖆𝖞𝖞, 𝗠𝗨𝗞𝗘𝗫 𝗦𝗧𝗢𝗥𝗬, 𝗦𝗮𝗿𝗲_𝗩𝗹𝗼𝗴𝘀𝘀𝘀𐙚, 𝙏𝙀𝙈𝘼𝙉 𝙈𝙊𝙇𝙀 𝙄𝙉𝘿, 𝙒𝙝𝙞𝙯 𝙄𝘿 𝘾𝙝𝙖𝙣𝙣𝙚𝙡, 𝙺𝙰𝙽𝙶 𝙵𝙰𝙳𝙾𝙻𝙸 𝙱𝙰𝙽𝚃𝙴𝙽, 𝚍𝚒𝚜𝚔𝚘𝚗𝚋𝚘𝚡, 𝚣𝚘𝚗𝚊 𝚜𝚑𝚘𝚛𝚝𝚜]",59
sorottanahair,"[SOROT TANAH AIR, Sorot Tanah Air]",23
dialectique,"[DIALECTIQUE, Dialectique]",21
totalpolitik,"[Total Politik, totalpolitik]",18
dekade08,"[@dekade_08, DEKADE 08, Dekade 08, dekade_08]",14
katadataindonesia,"[KATADATA Indonesia, Katadata Indonesia]",12
maswanto,"[MAS WANTO, Mas Wanto]",12
tvrinasional,"[@TVRINasional, TVRI Nasional, tvrinasional]",10
ahmadkhozinudin,"[AHMAD KHOZINUDIN, Ahmad Khozinudin]",9


[INFO] 13816 retweet, 4073 di antaranya ketemu post aslinya di data
[OK] 95 nodes, 77 edges, 30 KOL
[INFO] Edge per interaksi:
interaction
mention    50
reply      16
retweet    12
Name: Weight, dtype: int64
[INFO] 52% target edge juga muncul sebagai Author di data. Kalau mendekati 0%, cek apakah kolom Author berisi username (bukan nama tampilan).

=== KOL: agenda_pertemuan ===


,rank,author_id,is_media,jumlah_post,total_engagement,followers,pagerank,eigenvector,volume_score,network_score,score
0,1,@embah72,False,2.0,4.0,16039.0,0.193717,0.000000,0.640666,0.500000,0.570333
1,2,@thy_freond,False,1.0,0.0,11.0,0.104712,0.707107,0.300008,0.770271,0.535140
2,3,komisarisnetzen,False,0.0,0.0,3517.0,0.149215,0.500000,0.000000,0.738689,0.369344
3,4,prabowo,False,0.0,0.0,0.0,0.149215,0.500000,0.000000,0.738689,0.369344
4,5,@TheRubberDuck79,False,1.0,35.0,198916.0,0.000000,0.000000,0.700000,0.000000,0.350000
5,6,inckerala,False,0.0,0.0,0.0,0.193717,0.000000,0.000000,0.500000,0.250000
6,7,troublemakerid,False,1.0,0.0,1562.0,0.000000,0.000000,0.301178,0.000000,0.150589
7,8,@bambangpe,False,1.0,0.0,167.0,0.000000,0.000000,0.300126,0.000000,0.150063
8,9,@babagajah46,False,1.0,0.0,19.0,0.000000,0.000000,0.300014,0.000000,0.150007
9,10,Meta AI,False,1.0,0.0,0.0,0.000000,0.000000,0.300000,0.000000,0.150000



=== KOL: kerja_sama_bilateral ===


...

,rank,author_id,is_media,jumlah_post,total_engagement,followers,pagerank,eigenvector,volume_score,network_score,score
10,1,@sibambaang,False,1.0,20930.0,10834.0,0.073231,0.000000,0.352357,0.500000,0.426179
11,2,@oconXBoyy,False,1.0,7.0,3583.0,0.012005,0.700756,0.100863,0.581965,0.341414
12,3,nan,False,6.0,0.0,0.0,0.000000,0.000000,0.600000,0.000000,0.300000
13,4,thisismeonx,False,0.0,0.0,0.0,0.063027,0.000000,0.000000,0.430327,0.215164
14,5,Meta AI,False,2.0,0.0,689431.0,0.000000,0.000000,0.350000,0.000000,0.175000
15,6,@GlendMarti35542,False,2.0,0.0,1103.0,0.012005,0.086785,0.200240,0.143887,0.172064
16,7,prabowo,False,0.0,0.0,0.0,0.030154,0.155769,0.000000,0.317026,0.158513
17,8,@LambeSahamjja,False,1.0,41.0,84045.0,0.013280,0.014576,0.118775,0.101074,0.109925
18,9,@salman57701,False,1.0,0.0,36.0,0.012005,0.026923,0.100008,0.101174,0.100591
19,10,@doyreeves,False,1.0,0.0,146.0,0.012005,0.026163,0.100032,0.100633,0.100332



=== Nodes (urut PageRank) ===


,Id,Label,in_weight,out_weight,pagerank,eigenvector,followers,is_media,kol_theme,kol_score,is_kol
0,thisismeonx,thisismeonx,9,0,0.074881,0.000000,NaN,False,kerja_sama_bilateral,0.500000,True
1,sibambaang,@sibambaang,6,0,0.052806,0.000000,10857.0,False,dampak_geopolitik,0.426179,True
2,mhuseinali,@mhuseinali,1,0,0.034777,0.000000,24541.0,False,kerja_sama_bilateral,0.228267,False
3,admkuznetsov45,@admkuznetsov45,3,1,0.030731,0.000000,1158.0,False,kerja_sama_bilateral,0.501835,True
4,prabowo,prabowo,6,0,0.025423,0.161168,NaN,False,agenda_pertemuan,0.369344,True
5,embah72,@embah72,1,0,0.016015,0.000000,16072.0,False,agenda_pertemuan,0.570333,True
6,theeconomicinfluenceid,theeconomicinfluenceid,1,0,0.016015,0.000000,NaN,False,kerja_sama_bilateral,0.105113,False
7,youtube,youtube,1,0,0.016015,0.000000,NaN,False,dampak_geopolitik,0.075819,False
8,pororo266,@Pororo266,1,0,0.016015,0.000000,166.0,False,kerja_sama_bilateral,0.488466,True
9,inckerala,inckerala,1,0,0.016015,0.000000,NaN,False,agenda_pertemuan,0.250000,True



=== Edges (urut Weight) ===


,Source,Target,interaction,theme,Weight
0,glendmarti35542,prabowo,mention,dampak_geopolitik,2
1,4rsyat4ri,thisismeonx,reply,kerja_sama_bilateral,1
2,adindahan1,sibambaang,retweet,dampak_geopolitik,1
3,admkuznetsov45,mhuseinali,reply,kerja_sama_bilateral,1
4,3dis4,embah72,retweet,agenda_pertemuan,1
5,aliam36,youtube,mention,dampak_geopolitik,1
6,alqossam3,admkuznetsov45,retweet,kerja_sama_bilateral,1
7,babingepet01nk,admkuznetsov45,retweet,kerja_sama_bilateral,1
8,bangbatu524917,sibambaang,retweet,dampak_geopolitik,1
9,citraayuning7,thisismeonx,reply,kerja_sama_bilateral,1


# Step 6A - Load IndoBERT sentimen

In [41]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

indobert_tokenizer = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
indobert_model = AutoModelForSequenceClassification.from_pretrained(INDOBERT_MODEL_NAME).eval()

# label map dari config model, bukan hardcode (urutan label tiap model beda)
LABEL_NORMALIZE = {"positive": "positif", "negative": "negatif", "neutral": "netral"}
INDOBERT_LABEL_MAP = {int(i): LABEL_NORMALIZE.get(str(l).lower(), str(l).lower())
                      for i, l in indobert_model.config.id2label.items()}
print("[INFO] Label map dari config model:", INDOBERT_LABEL_MAP)
if not {"positif", "negatif", "netral"} <= set(INDOBERT_LABEL_MAP.values()):
    raise ValueError("id2label tidak dikenali (mis. LABEL_0). Cek model card, isi INDOBERT_LABEL_MAP manual.")


def indobert_predict_sentiment(text):
    inputs = indobert_tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        probs = torch.softmax(indobert_model(**inputs).logits, dim=1)[0]
    pred_id = int(torch.argmax(probs))
    return INDOBERT_LABEL_MAP[pred_id], float(probs[pred_id])

print("IndoBERT siap.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[INFO] Label map dari config model: {0: 'positif', 1: 'netral', 2: 'negatif'}
IndoBERT siap.


# Step 6B - IndoBERT per post, lalu digabung per author

In [42]:
def aggregate_author_sentiment(labels):
    """kontroversi = campuran positif & negatif signifikan. netral = mayoritas netral / tanpa sikap."""
    counts = Counter(labels)
    pos, neg, net = (counts[k] / len(labels) for k in ("positif", "negatif", "netral"))
    if pos >= 0.3 and neg >= 0.3:
        return "kontroversi", max(pos, neg)
    if net >= 0.5 or pos == neg:
        return "netral", net
    return ("positif", pos) if pos > neg else ("negatif", neg)


def label_sentiment_batch(items):
    """items: list of (text, theme). Return label per item (None kalau gagal)."""
    numbered = "\n".join(f'{i}. Tema: {th.replace("_", " ")}\n   Teks: "{t}"'
                         for i, (t, th) in enumerate(items, 1))
    prompt = f"""Tentukan sikap penulis setiap cuitan terhadap tema yang disebut:
positif (mendukung, memuji, optimis), negatif (mengkritik, menolak, pesimis),
atau netral (informasi tanpa sikap, atau sikap tidak jelas).

Item:
{numbered}

Jawab HANYA JSON array berisi tepat {len(items)} objek dengan field index dan label."""
    n = len(items)
    labels = ask_json(prompt, parse=lambda p: [x.get("label") for x in order_by_index(p, n)]) or [None] * n
    return [lab if lab in ("positif", "negatif", "netral") else None for lab in labels]


def kol_posts(key, theme):
    """Post asli akun ini (semua ejaan) di tema ini, paling berpengaruh duluan. Tidak ada -> teks yang me-retweet dia."""
    own = (df_original[(df_original["author_key"] == key) & (df_original["theme"] == theme)]
           .sort_values("engagement", ascending=False)["text_clean"].dropna().tolist())
    return (own or rt_posts.get((key, theme), []))[:MAX_POSTS_PER_AUTHOR_SUMMARY]


author_sentiment_results = {}   # (Id, theme) -> hasil IndoBERT; Id = key akun dari Step 5B
review_queue = {}   # (text, theme) -> (confidence, reason), dedup otomatis

for key, theme in zip(df_top_authors["Id"], df_top_authors["theme"]):
    posts = kol_posts(key, theme)
    if not posts:
        continue

    post_sentiments = [indobert_predict_sentiment(t) for t in posts]
    for text, (_, conf) in zip(posts, post_sentiments):
        if conf < SENTIMENT_REVIEW_THRESHOLD:
            review_queue.setdefault((text, theme), (conf, "low_conf"))
        elif random.random() < REVIEW_SAMPLE_RATE:
            review_queue.setdefault((text, theme), (conf, "sample"))

    label, confidence = aggregate_author_sentiment([lab for lab, _ in post_sentiments])
    author_sentiment_results[(key, theme)] = {
        "sentiment": label, "confidence": confidence, "post_sentiments": post_sentiments, "posts": posts,
    }

print(f"[INFO] IndoBERT selesai prediksi {len(author_sentiment_results)} KOL")

# label koreksi per post dari AI -> training data sentimen
sentiment_training_data = []
queue_items = list(review_queue.items())
print(f"[INFO] {len(queue_items)} post dikirim ke AI untuk label sentimen")

for b in range(0, len(queue_items), BATCH_SIZE_CLASSIFY):
    chunk = queue_items[b:b + BATCH_SIZE_CLASSIFY]
    try:
        labels = label_sentiment_batch([key for key, _ in chunk])
    except Exception as e:
        if not is_quota_error(e):
            raise
        print(f"[STOP] Kuota habis saat label sentimen. {len(sentiment_training_data)} label tersimpan.")
        break
    sentiment_training_data += [
        {"text": text, "theme": theme, "label": label, "label_source": "gemini",
         "review_reason": reason, "model_version": INDOBERT_MODEL_NAME, "last_confidence": conf}
        for ((text, theme), (conf, reason)), label in zip(chunk, labels) if label
    ]
    time.sleep(SLEEP_BETWEEN_CALLS)

save_pending("sentimen", sentiment_training_data)

[INFO] IndoBERT selesai prediksi 25 KOL
[INFO] 5 post dikirim ke AI untuk label sentimen
[pending] sentimen: 5 baris ditulis ke /content/drive/MyDrive/pending_sentiment_data.jsonl


# Step 6C - Sentimen final (Colab AI)

In [43]:
SENTIMENT_LABELS = ("positif", "negatif", "netral", "kontroversi")
SENTIMENT_FALLBACK = {"sentiment": None, "reason": "error API"}


def judge_sentiment(author_label, theme, texts, indobert_label):
    """Colab AI: sentimen final akun terhadap tema (summary dibuat LLM lokal di Step 7B)."""
    joined = "\n".join(f"- {t}" for t in texts)
    prompt = f"""Cuitan dari akun "{author_label}" tentang tema "{theme.replace('_', ' ')}" (topik: {TOPIC}):

{joined}

Tentukan SIKAP akun ini terhadap topik/tema tersebut, BUKAN nada bahasanya:
- positif: mendukung, memuji, atau membela
- negatif: mengkritik, menolak, atau menyindir
- netral: hanya menyampaikan informasi, tanpa sikap yang jelas
- kontroversi: dukungan dan kritik sama kuat
Contoh: membela Prabowo sambil memaki pengkritiknya = positif (kasar ke pihak lain tidak mengubah sikapnya ke topik).

Model otomatis (IndoBERT) menebak "{indobert_label}". Pakai sebagai petunjuk saja, koreksi kalau salah.

Jawab HANYA JSON dengan field:
- sentiment: salah satu dari positif, negatif, netral, kontroversi
- reason: 1 kalimat alasan label itu berdasarkan isi teks"""
    return ask_json(prompt, retry=3) or SENTIMENT_FALLBACK


results = []
quota_hit = False

for _, row in df_top_authors.iterrows():
    data = author_sentiment_results.get((row["Id"], row["theme"]))
    if data is None:
        continue
    print(f"Sentimen final {row['author_id']} | tema={row['theme']} | IndoBERT={data['sentiment']} ...")

    try:
        judged = judge_sentiment(row["author_id"], row["theme"], data["posts"], data["sentiment"])
    except Exception as e:
        if not is_quota_error(e):
            raise
        quota_hit = True
        break

    # label AI dipakai kalau valid; kalau AI gagal, tetap pakai IndoBERT
    ai_label = judged.get("sentiment")
    final = ai_label if ai_label in SENTIMENT_LABELS else data["sentiment"]
    source = "indobert" if final == data["sentiment"] else "ai_koreksi"
    if source == "ai_koreksi":
        print(f"  [KOREKSI] {data['sentiment']} -> {final}: {judged.get('reason', '')}")

    results.append({
        "theme": row["theme"],
        "rank": int(row["rank"]),
        "author_id": row["author_id"],
        "Id": row["Id"],
        "score": round(row["score"], 2),
        "volume_score": round(row["volume_score"], 2),
        "network_score": round(row["network_score"], 2),
        "pagerank": round(row["pagerank"], 4),
        "eigenvector": round(row["eigenvector"], 4),
        "sentiment": final,
        "sentiment_source": source,
        "sentiment_indobert": data["sentiment"],
        "indobert_share": round(data["confidence"], 2),   # porsi post yang sesuai label IndoBERT, bukan confidence AI
        "sentiment_reason": judged.get("reason", ""),
    })

df_final_sentiment = pd.DataFrame(results)

if quota_hit:
    print(f"\n[STOP] Berhenti di tengah karena kuota habis. {len(results)} KOL sudah beres. "
          "Tunggu beberapa menit lalu jalankan ulang cell ini sebelum lanjut ke Step 7.")
else:
    print(f"\n[DONE] Sentimen final semua KOL beres. Total API call sesi ini: {api_call_count}")
if len(df_final_sentiment):
    print(f"[INFO] {(df_final_sentiment['sentiment_source'] == 'ai_koreksi').sum()} dari {len(df_final_sentiment)} "
          "label IndoBERT dikoreksi AI")
display(df_final_sentiment)

Sentimen final @embah72 | tema=agenda_pertemuan | IndoBERT=negatif ...
Sentimen final @thy_freond | tema=agenda_pertemuan | IndoBERT=negatif ...
Sentimen final @TheRubberDuck79 | tema=agenda_pertemuan | IndoBERT=netral ...
Sentimen final inckerala | tema=agenda_pertemuan | IndoBERT=netral ...
Sentimen final troublemakerid | tema=agenda_pertemuan | IndoBERT=netral ...
  [KOREKSI] netral -> positif: Cuitan ini secara konsisten menggunakan narasi yang memuji dan mendukung, menggambarkan kunjungan Presiden Prabowo sebagai 'strategis', beliau sebagai 'tamu kehormatan utama', 'disambut hangat', dan pidatonya sebagai 'tegas' yang menekankan kesejahteraan rakyat.
Sentimen final @bambangpe | tema=agenda_pertemuan | IndoBERT=netral ...
Sentimen final @babagajah46 | tema=agenda_pertemuan | IndoBERT=netral ...
  [KOREKSI] netral -> positif: Cuitan ini menggunakan diksi yang sangat positif seperti 'momen akrab', 'bermartabat', dan 'setara di panggung dunia' untuk menggambarkan kunjungan dan pertemu

,theme,rank,author_id,Id,score,volume_score,network_score,pagerank,eigenvector,sentiment,sentiment_source,sentiment_indobert,indobert_share,sentiment_reason
0,agenda_pertemuan,1,@embah72,embah72,0.57,0.64,0.50,0.1937,0.0000,negatif,indobert,negatif,1.0,"Akun ini secara eksplisit mengkritik dan merendahkan Prabowo, menyebutnya 'blo'on' dan 'kosong kepalanya' serta menyindir kebijakannya."
1,agenda_pertemuan,2,@thy_freond,thyfreond,0.54,0.30,0.77,0.1047,0.7071,negatif,indobert,negatif,1.0,"Cuitan tersebut secara eksplisit menyatakan 'malu', menuduh adanya 'blunder' dan 'tidak nyambung' dalam pertemuan dengan Rusia, serta mempertanyakan kapabilitas pemimpin, yang semuanya menunjukkan kritik terhadap topik."
2,agenda_pertemuan,5,@TheRubberDuck79,therubberduck79,0.35,0.70,0.00,0.0000,0.0000,netral,indobert,netral,1.0,"Teks ini hanya menyajikan daftar faktual peserta dan perwakilan dari setiap negara dalam sebuah KTT BRICS, tanpa adanya opini, pujian, kritik, atau pembelaan terhadap topik tersebut."
3,agenda_pertemuan,6,inckerala,inckerala,0.25,0.00,0.50,0.1937,0.0000,netral,indobert,netral,1.0,"Cuitan ini hanya menampilkan daftar negara, pemimpin, dan bendera tanpa ekspresi pendapat, dukungan, atau kritik terhadap topik kunjungan Prabowo atau pertemuan dengan Putin."
4,agenda_pertemuan,7,troublemakerid,troublemakerid,0.15,0.30,0.00,0.0000,0.0000,positif,ai_koreksi,netral,1.0,"Cuitan ini secara konsisten menggunakan narasi yang memuji dan mendukung, menggambarkan kunjungan Presiden Prabowo sebagai 'strategis', beliau sebagai 'tamu kehormatan utama', 'disambut hangat', dan pidatonya sebagai 'tegas' yang menekankan kesejahteraan rakyat."
5,agenda_pertemuan,8,@bambangpe,bambangpe,0.15,0.30,0.00,0.0000,0.0000,netral,indobert,netral,1.0,"Cuitan ini hanya memberitahukan adanya 'Comparison Statement Analysis' terkait kunjungan Presiden RI yang dimuat di dua media, tanpa memberikan opini atau sikap pribadi terhadap topik tersebut."
6,agenda_pertemuan,9,@babagajah46,babagajah46,0.15,0.30,0.00,0.0000,0.0000,positif,ai_koreksi,netral,1.0,"Cuitan ini menggunakan diksi yang sangat positif seperti 'momen akrab', 'bermartabat', dan 'setara di panggung dunia' untuk menggambarkan kunjungan dan pertemuan tersebut, menunjukkan dukungan dan apresiasi yang kuat."
7,agenda_pertemuan,10,Meta AI,metaai,0.15,0.30,0.00,0.0000,0.0000,positif,indobert,positif,1.0,"Cuitan ini menyoroti pola pendampingan Seskab Teddy yang sangat konsisten dan intensif untuk Prabowo, membandingkannya dengan pola Jokowi yang lebih selektif, yang secara implisit menonjolkan kekhasan dan potensi efisiensi dalam gaya kerja Prabowo."
8,dampak_geopolitik,1,@sibambaang,sibambaang,0.43,0.35,0.50,0.0732,0.0000,positif,ai_koreksi,negatif,1.0,"Tweet ini secara bangga menyebutkan para pemimpin Indonesia yang baru/akan menjabat (termasuk Prabowo dan Gibran) setelah memaparkan krisis geopolitik global yang parah, mengimplikasikan kepercayaan atau afirmasi terhadap kepemimpinan mereka di tengah situasi sulit ini."
9,dampak_geopolitik,2,@oconXBoyy,oconxboyy,0.34,0.10,0.58,0.0120,0.7008,negatif,indobert,negatif,1.0,"Cuitan ini secara terang-terangan merendahkan dan mengejek peran Prabowo dalam pertemuan tersebut dengan kata-kata seperti 'dinasihati', 'cuma plonga-plongo', dan 'cuuiihh!!'."


# Step 7A : load LLM lokal


In [44]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

if not torch.cuda.is_available():
    print("⚠️ Tidak ada GPU: LLM 9B di CPU bisa beberapa menit per KOL. "
          "Runtime -> Change runtime type -> T4 GPU, lalu mulai lagi dari cell Install.")

# cell ini dijalankan ulang -> pakai model yang sudah ada; load kedua kali = 2x memori GPU (bisa out of memory)
if "llm" in globals():
    print(f"[INFO] LLM lokal sudah ter-load, dipakai ulang: {LLM_MODEL_FILE}")
else:
    # download ~5,8 GB sekali per session (tersimpan di cache Colab selama session masih hidup)
    model_path = hf_hub_download(repo_id=LLM_MODEL_REPO, filename=LLM_MODEL_FILE)
    llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=LLM_N_CTX, n_batch=512, verbose=False)
    print(f"[OK] LLM lokal siap: {LLM_MODEL_FILE}")

[INFO] LLM lokal sudah ter-load, dipakai ulang: gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf


# Step 7B: summary + stance oposisi / moderat / supporter

In [45]:
LLM_TEST_N = None   # angka = coba beberapa KOL dulu; None = semua KOL

STANCES = ("oposisi", "moderat", "supporter")
UNCLASSIFIED = "tidak_terklasifikasi"
STANCE_FIELDS = ["relevan", "summary", "stance", "stance_reason"]
STANCE_SCHEMA = {
    "type": "object",
    "properties": {"relevan": {"type": "boolean"},
                   "summary": {"type": "string"},
                   "stance": {"type": "string", "enum": list(STANCES)},
                   "stance_reason": {"type": "string"}},
    "required": STANCE_FIELDS,
}

STANCE_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}

Cuitan dari akun "{author}" tentang tema ini:
{posts}

Sentimen final akun ini terhadap tema: {sentiment}
Alasan sentimen: {sentiment_reason}

Tugas:
0. relevan: true kalau cuitan benar-benar membahas topik di atas; false kalau membahas peristiwa lain
   dan hanya kebetulan menyebut nama tokoh, negara, atau acara yang sama.
1. summary: 2-3 kalimat tentang pandangan atau narasi utama akun ini soal tema tersebut.
   Jelaskan isinya, jangan menyalin cuitan kata per kata.
2. stance: sikap akun ini terhadap {target}, pilih satu:
   - supporter: mendukung, memuji, atau membela {target}
   - oposisi: mengkritik, menolak, atau menyerang {target}
   - moderat: hanya menyampaikan informasi, seimbang, atau kritik yang tidak menolak {target}
   Jadikan sentimen final sebagai sinyal utama, tapi putuskan dari isi cuitan:
   sentimen positif terhadap tema belum tentu berarti mendukung {target}, dan sebaliknya.
3. stance_reason: 1 kalimat alasan stance terhadap {target} berdasarkan isi cuitan.
   Jangan mengulang alasan sentimen di atas kata per kata.

Jawab dalam bahasa Indonesia. Output HANYA JSON."""


def fmt_posts(posts, max_words=120):
    return "\n".join(f"{i}. " + " ".join(str(p).split()[:max_words]) for i, p in enumerate(posts, 1))


def summarize_stance(row, posts):
    """Satu panggilan LLM lokal -> {relevan, summary, stance, stance_reason}. Stance di luar 3 label tidak ditebak."""
    content = STANCE_PROMPT.format(
        topic=TOPIC, theme=row["theme"].replace("_", " "), target=STANCE_TARGET, author=row["author_id"],
        posts=fmt_posts(posts), sentiment=row["sentiment"], sentiment_reason=row["sentiment_reason"] or "-",
    )
    r = llm.create_chat_completion(
        messages=[{"role": "user", "content": content}],   # Gemma tidak punya role 'system'
        response_format={"type": "json_object", "schema": STANCE_SCHEMA},
        temperature=0.2, max_tokens=600,
    )
    out = json.loads(r["choices"][0]["message"]["content"])
    stance = str(out.get("stance", "")).strip().lower()
    return {"relevan": bool(out.get("relevan", True)),
            "summary": str(out.get("summary") or "").strip(),
            "stance": stance if stance in STANCES else UNCLASSIFIED,
            "stance_reason": str(out.get("stance_reason") or "").strip()}


if len(df_final_sentiment) < len(author_sentiment_results):
    print(f"[WARN] Sentimen final baru ada untuk {len(df_final_sentiment)} dari {len(author_sentiment_results)} KOL "
          "(Step 6C berhenti karena kuota). Jalankan ulang Step 6C dulu kalau mau hasil lengkap.")

todo = df_final_sentiment if LLM_TEST_N is None else df_final_sentiment.head(LLM_TEST_N)
records = []
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Summary + stance (LLM lokal)"):
    posts = author_sentiment_results[(row["Id"], row["theme"])]["posts"]
    try:
        out = summarize_stance(row, posts)
    except Exception as e:
        print(f"[ERROR] {row['author_id']} | {row['theme']}: {type(e).__name__}: {e}")
        out = {"relevan": None, "summary": "", "stance": UNCLASSIFIED,
               "stance_reason": f"error LLM: {type(e).__name__}"}
    records.append({**row.to_dict(), **out})

df_summary = pd.DataFrame(records)

print(f"[OK] Summary + stance selesai untuk {len(df_summary)} KOL")
off_topic = df_summary["relevan"].eq(False)
if off_topic.any():
    print(f"[WARN] {off_topic.sum()} KOL dinilai TIDAK membahas topik (salah tema di Step 4D): "
          f"{', '.join(df_summary.loc[off_topic, 'author_id'].astype(str))}. Tidak dihitung di rekap & ringkasan tema.")
if LLM_TEST_N is not None:
    print(f"[TEST] Baru {len(df_summary)} KOL. Kalau hasilnya sudah pas, set LLM_TEST_N = None lalu jalankan ulang cell ini.")

print("\n=== Summary & stance per KOL ===")
display(df_summary[["theme", "rank", "author_id", "relevan", "score", "volume_score", "network_score", "pagerank", "eigenvector",
                    "sentiment", "sentiment_source", "stance", "summary", "stance_reason", "sentiment_reason"]])

print(f"\n=== Rekap stance terhadap {STANCE_TARGET} per tema ===")
on_topic = df_summary[df_summary["relevan"].ne(False)]
stance_counts = pd.crosstab(on_topic["theme"], on_topic["stance"]).reindex(columns=[*STANCES, UNCLASSIFIED], fill_value=0)
display(stance_counts.loc[:, stance_counts.any()].assign(total=stance_counts.sum(axis=1)))

Summary + stance (LLM lokal):   0%|          | 0/25 [00:00<?, ?it/s]

[OK] Summary + stance selesai untuk 25 KOL

=== Summary & stance per KOL ===


,theme,rank,author_id,relevan,score,volume_score,network_score,pagerank,eigenvector,sentiment,sentiment_source,stance,summary,stance_reason,sentiment_reason
0,agenda_pertemuan,1,@embah72,True,0.57,0.64,0.50,0.1937,0.0000,negatif,indobert,oposisi,"Akun ini mengkritik keras kunjungan Prabowo ke Rusia dan pertemuannya dengan Putin, menyebut Prabowo 'blo'on' dan 'kosong kepalanya' serta menyindir kebijakannya yang dianggap tidak bijaksana.","Akun ini secara eksplisit mengkritik dan merendahkan Prabowo, menyebutnya 'blo'on' dan 'kosong kepalanya' serta menyindir kebijakannya.","Akun ini secara eksplisit mengkritik dan merendahkan Prabowo, menyebutnya 'blo'on' dan 'kosong kepalanya' serta menyindir kebijakannya."
1,agenda_pertemuan,2,@thy_freond,True,0.54,0.30,0.77,0.1047,0.7071,negatif,indobert,oposisi,"Akun ini mengkritik Presiden Prabowo karena dianggap tidak mampu dalam pertemuan dengan Rusia, dengan menyebutnya 'blunder' dan 'tidak nyambung'.",Akun ini secara eksplisit menyatakan rasa malu dan mengkritik kinerja Presiden Prabowo dalam pertemuan dengan Rusia.,"Cuitan tersebut secara eksplisit menyatakan 'malu', menuduh adanya 'blunder' dan 'tidak nyambung' dalam pertemuan dengan Rusia, serta mempertanyakan kapabilitas pemimpin, yang semuanya menunjukkan kritik terhadap topik."
2,agenda_pertemuan,5,@TheRubberDuck79,True,0.35,0.70,0.00,0.0000,0.0000,netral,indobert,moderat,"Cuitan ini menyajikan daftar faktual peserta dan perwakilan dari setiap negara dalam KTT BRICS 2026 di New Delhi, India, tanpa adanya opini atau komentar terkait agenda pertemuan Presiden Prabowo dengan Vladimir Putin.",Cuitan ini hanya menyajikan informasi faktual tanpa adanya opini atau komentar yang mendukung atau menyerang Presiden Prabowo dan pemerintah.,"Teks ini hanya menyajikan daftar faktual peserta dan perwakilan dari setiap negara dalam sebuah KTT BRICS, tanpa adanya opini, pujian, kritik, atau pembelaan terhadap topik tersebut."
3,agenda_pertemuan,6,inckerala,True,0.25,0.00,0.50,0.1937,0.0000,netral,indobert,moderat,"Cuitan ini hanya menampilkan daftar negara, pemimpin, dan bendera tanpa ekspresi pendapat, dukungan, atau kritik terhadap topik kunjungan Prabowo atau pertemuan dengan Putin.","Cuitan ini hanya menampilkan daftar negara, pemimpin, dan bendera tanpa ekspresi pendapat, dukungan, atau kritik terhadap topik kunjungan Prabowo atau pertemuan dengan Putin.","Cuitan ini hanya menampilkan daftar negara, pemimpin, dan bendera tanpa ekspresi pendapat, dukungan, atau kritik terhadap topik kunjungan Prabowo atau pertemuan dengan Putin."
4,agenda_pertemuan,7,troublemakerid,True,0.15,0.30,0.00,0.0000,0.0000,positif,ai_koreksi,supporter,"Cuitan ini memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, menyebutnya sebagai kunjungan strategis dan pidato Presiden Prabowo sebagai tegas dalam menekankan kesejahteraan rakyat.",Cuitan ini secara konsisten menggunakan narasi yang memuji dan mendukung Presiden Prabowo dan pemerintah.,"Cuitan ini secara konsisten menggunakan narasi yang memuji dan mendukung, menggambarkan kunjungan Presiden Prabowo sebagai 'strategis', beliau sebagai 'tamu kehormatan utama', 'disambut hangat', dan pidatonya sebagai 'tegas' yang menekankan kesejahteraan rakyat."
5,agenda_pertemuan,8,@bambangpe,True,0.15,0.30,0.00,0.0000,0.0000,netral,indobert,moderat,"Akun ini membagikan informasi tentang analisis pernyataan Presiden RI yang dimuat di dua media asing, The Star dan The Straits Times, terkait kunjungannya ke Vladivostok, Rusia.",Akun ini hanya membagikan informasi tentang analisis pernyataan Presiden RI tanpa memberikan opini atau sikap pribadi terhadap topik tersebut.,"Cuitan ini hanya memberitahukan adanya 'Comparison Statement Analysis' terkait kunjungan Presiden RI yang dimuat di dua media, tanpa memberikan opini atau sikap pribadi terhadap topik tersebut."
6,agenda_pertemuan,9,@babagajah46,True,0.15,0.30,0.00,0.0000,0.0000,positif,ai_koreksi,supporter,"Akun ini memuji kunjungan Presiden Prabowo ke Vla


=== Rekap stance terhadap Presiden Prabowo dan pemerintah per tema ===


stance,oposisi,moderat,supporter,total
theme,,,,
agenda_pertemuan,2,4,2,8
dampak_geopolitik,4,2,2,8
kerja_sama_bilateral,0,3,6,9


# Step 7C - Ringkasan per tema

In [46]:
THEME_FIELDS = ["ringkasan", "narasi_supporter", "narasi_moderat", "narasi_oposisi"]
THEME_SCHEMA = {"type": "object", "properties": {f: {"type": "string"} for f in THEME_FIELDS}, "required": THEME_FIELDS}

THEME_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}
Komposisi sikap KOL di tema ini: {composition}

Ringkasan tiap KOL, urut dari yang paling berpengaruh:
{kols}

Tugas (HANYA berdasarkan ringkasan di atas, jangan menambah fakta dari luar):
1. ringkasan: 3-4 kalimat tentang apa yang dibicarakan para KOL di tema ini dan ke mana arah percakapannya.
2. narasi_supporter: 1-2 kalimat argumen utama kubu supporter. Tulis "-" kalau tidak ada supporter.
3. narasi_moderat: 1-2 kalimat argumen utama kubu moderat. Tulis "-" kalau tidak ada.
4. narasi_oposisi: 1-2 kalimat argumen utama kubu oposisi. Tulis "-" kalau tidak ada.

Jawab dalam bahasa Indonesia. Output HANYA JSON."""


def summarize_theme(theme, kols):
    """Satu panggilan LLM lokal per tema, dari hasil Step 7B (bukan cuitan mentah, supaya prompt tetap pendek)."""
    counts = kols["stance"].value_counts()
    content = THEME_PROMPT.format(
        topic=TOPIC, theme=theme.replace("_", " "), target=STANCE_TARGET,
        composition=", ".join(f"{s} {counts.get(s, 0)}" for s in STANCES),
        kols="\n".join(f"- [{r.stance}] {r.author_id}: {r.summary}" for r in kols.itertuples()),
    )
    r = llm.create_chat_completion(
        messages=[{"role": "user", "content": content}],
        response_format={"type": "json_object", "schema": THEME_SCHEMA},
        temperature=0.2, max_tokens=700,
    )
    out = json.loads(r["choices"][0]["message"]["content"])
    return {f: str(out.get(f) or "").strip() for f in THEME_FIELDS}


theme_records = []
usable = df_summary[(df_summary["summary"] != "") & df_summary["relevan"].ne(False)]   # KOL salah tema tidak ikut
for theme, kols in usable.sort_values("rank").groupby("theme"):
    try:
        out = summarize_theme(theme, kols)
    except Exception as e:
        print(f"[ERROR] tema {theme}: {type(e).__name__}: {e}")
        out = dict.fromkeys(THEME_FIELDS, "")
    counts = kols["stance"].value_counts()
    theme_records.append({"theme": theme, "jumlah_kol": len(kols),
                          **{s: int(counts.get(s, 0)) for s in STANCES},
                          "kol_teratas": ", ".join(kols["author_id"].head(3)), **out})

df_theme_summary = pd.DataFrame(theme_records)
print(f"[OK] Ringkasan {len(df_theme_summary)} tema selesai")
if LLM_TEST_N is not None:
    print("[TEST] Ringkasan tema ini baru dari KOL mode tes. Jalankan ulang setelah Step 7B dengan LLM_TEST_N = None.")
display(df_theme_summary)

[OK] Ringkasan 3 tema selesai


,theme,jumlah_kol,oposisi,moderat,supporter,kol_teratas,ringkasan,narasi_supporter,narasi_moderat,narasi_oposisi
0,agenda_pertemuan,8,2,4,2,"@embah72, @thy_freond, @TheRubberDuck79","Para KOL di media sosial Indonesia membahas kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin. Percakapan ini didominasi oleh kritik dari kubu oposisi yang menyebut kunjungan tersebut tidak bijaksana dan tidak efektif, sementara kubu supporter memuji kunjungan tersebut sebagai langkah strategis yang menunjukkan martabat Indonesia di panggung dunia.",Kubu supporter memuji kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin sebagai langkah strategis yang menunjukkan martabat Indonesia di panggung dunia.,Kubu moderat menyajikan informasi faktual tentang kunjungan tersebut tanpa memberikan opini atau komentar yang jelas.,"Kubu oposisi mengkritik keras kunjungan Prabowo ke Rusia dan pertemuannya dengan Putin, menyebutnya tidak bijaksana dan tidak efektif."
1,dampak_geopolitik,8,4,2,2,"@sibambaang, @oconXBoyy, nan","Para KOL membahas kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Putin, serta dampaknya terhadap geopolitik Indonesia. Percakapan ini terbagi menjadi tiga kubu: supporter yang memuji Prabowo, oposisi yang mengkritiknya, dan moderat yang membahas isu-isu terkait dengan netralitas dan kebijakan energi.",Kubu supporter memuji Prabowo sebagai pemimpin yang berani dan tegas dalam menghadapi tantangan geopolitik global.,Kubu moderat membahas dampak kunjungan Prabowo terhadap geopolitik Indonesia dan kebijakan pemerintah dalam meningkatkan kemandirian energi.,Kubu oposisi mengkritik Prabowo karena dianggap tidak efektif dalam menangani isu geopolitik dan mempertanyakan kemampuan serta prioritasnya.
2,kerja_sama_bilateral,9,0,3,6,"@admkuznetsov45, @Pororo266, @nai_nisa72","Para KOL di tema ini membahas kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin, serta kerja sama bilateral antara kedua negara, terutama di sektor energi dan pangan. Percakapan cenderung positif, dengan banyak KOL yang memuji upaya pemerintah Indonesia untuk meningkatkan ketahanan energi dan ekonomi nasional.","Kubu supporter memuji kunjungan Presiden Prabowo ke Rusia dan pertemuannya dengan Putin sebagai langkah positif untuk meningkatkan ketahanan energi dan ekonomi nasional, serta membuka akses migas murah demi daya beli rakyat.","Kubu moderat menyampaikan informasi tentang rencana kerja sama bilateral antara Indonesia dan Rusia, termasuk proyek hilirisasi bauksit dan proyek pupuk, serta pertimbangan pemerintah Indonesia mencari pasokan minyak dari negara lain, termasuk Rusia.",-
